# **1. ETL**

## **1.1. Librerias**

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import geopandas as gpd

## **1.2. Funciones**

In [2]:
def limpiar_nombre(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = texto.strip()
    texto = texto.capitalize()
    return texto

In [3]:
def Resumen_Faltantes (df):
    faltantes = df.isnull().sum()
    porcentaje = (faltantes/len(df))*100
    
    tabla = pd.DataFrame({
        'faltantes':faltantes,
        'porcentaje':porcentaje
    })
    
    tabla = tabla[tabla['faltantes']>0]
    tabla = tabla.sort_values('porcentaje', ascending = False)
    tabla['porcentaje'] = tabla['porcentaje'].round(3)
    
    return tabla

In [4]:
def procesar_df(df):
    df = df.copy()

    # =========================
    # 1. LIMPIEZA DE COLUMNAS
    # =========================
    df.columns = df.columns.astype(str)

    df.columns = [
        col.encode('latin1', errors='ignore').decode('utf-8', errors='ignore')
        for col in df.columns
    ]

    df.columns = [
        unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
        for col in df.columns
    ]

    df.columns = [col.lower() for col in df.columns]

    # =========================
    # 2. RENOMBRAR COLUMNAS
    # =========================
    renombrar_columnas = {
        'nom_grupo': 'nombre_grupo',
        'nom_upgd': 'nombre_upgd',
        'nacionalid': 'nacionalidad',
        'confirmados': 'confirmado',
        'sem_ges': 'semana_ges'
    }

    df = df.rename(columns=renombrar_columnas)

    # =========================
    # 3. UNIFICAR DUPLICADAS
    # =========================
    df = df.T.groupby(level=0).first().T

    # =========================
    # 4. LIMPIAR VACÍOS
    # =========================
    df = df.replace(r'^\s*$', np.nan, regex=True)

    valores_nulos = [
        "NA", "N/A", "NULL", "null", "nan", "NaN",
        "SIN INFORMACION", "SIN INFORMACIÓN", "NO APLICA"
    ]
    df = df.replace(valores_nulos, np.nan)

    # =========================
    # 5. FECHAS
    # =========================
    cols_fecha = [c for c in df.columns if c.startswith(('fec_', 'fecha_', 'ini'))]

    for col in cols_fecha:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    # =========================
    # 6. FUNCIONES AUXILIARES
    # =========================
    def limpiar_texto(x):
        if pd.isna(x): return np.nan
        x = str(x).strip()
        if x == "" or x == "SIN INFORMACION": return np.nan
        x = unicodedata.normalize('NFKD', x).encode('ascii','ignore').decode('utf-8')
        return x.upper()

    def limpiar_pais(x):
        if pd.isna(x): return np.nan
        x = str(x).strip()
        if x == "": return np.nan
        x = unicodedata.normalize('NFKD', x).encode('ascii','ignore').decode('utf-8')
        return " ".join(x.split()).upper()

    def limpiar_nacionalidad(x):
        if pd.isna(x): return np.nan
        try:
            return f"{int(float(x)):03d}"
        except:
            return np.nan

    def limpiar_semana(x):
        try:
            x = int(float(x))
            return x if 1 <= x <= 53 else np.nan
        except:
            return np.nan

    def corregir_estrato(x):
        try:
            return str(int(float(x)))
        except:
            return np.nan

    # =========================
    # 7. DICCIONARIOS
    # =========================
    categorias = {
        "tip_cas": {"2":"Probable","3":"Confirmado Lab","4":"Confirmado Clínica","5":"Nexo","1":"Sospechoso"},
        "tip_ss": {"C":"Contributivo","S":"Subsidiado","P":"Excepción","E":"Especial","N":"No asegurado","I":"Indeterminado"},
        "uni_med": {"1":"Años","2":"Meses","3":"Días","4":"Horas","5":"Minutos","0":"No aplica"},
        "area": {"1":"Cabecera","2":"Centro Poblado","3":"Rural"},
        "ajuste": {"0":"No aplica","3":"Lab","4":"Clínica","5":"Nexo","6":"Descartado"},
        "fuente": {"1.0":"Rutina","2.0":"BAI","3.0":"Intensificada","4.0":"BAC","5.0":"Investigación","0.0":"No aplica"}
    }

    col_group_map = [
        "gp_discapa","gp_gestan","gp_indigen","gp_mad_com","gp_migrant",
        "gp_otros","gp_pobicfb","gp_psiquia","gp_vic_vio",
        "gp_desplaz","gp_desmovi","gp_carcela","pac_hos"
    ]

    # =========================
    # 8. APLICAR LIMPIEZAS
    # =========================
    if "estrato" in df.columns:
        df["estrato"] = df["estrato"].apply(corregir_estrato)

    if "nacionalidad" in df.columns:
        df["nacionalidad"] = df["nacionalidad"].apply(limpiar_nacionalidad)

    if "nombre_grupo" in df.columns:
        df["nombre_grupo"] = df["nombre_grupo"].apply(limpiar_texto)

    for col in ["pais_residencia", "pais_ocurrencia", "nombre_nacionalidad"]:
        if col in df.columns:
            df[col] = df[col].apply(limpiar_pais)

    if "semana_ges" in df.columns:
        df["semana_ges"] = df["semana_ges"].apply(limpiar_semana).astype("Int64")

    # =========================
    # 9. MAPEOS
    # =========================
    for col, mapa in categorias.items():
        if col in df.columns:
            df[col] = df[col].astype(str).map(mapa)

    if "confirmado" in df.columns:
        df["confirmado"] = df["confirmado"].astype(str).map({
            "0":"No confirmado","1":"Confirmado"
        })

    for col in col_group_map:
        if col in df.columns:
            df[col] = df[col].astype(str).map({"1":"Sí","2":"No"})

    # =========================
    # 10. TIPOS DE DATOS
    # =========================
    for col in ["ano","semana"]:
        if col in df.columns:
            df[col] = df[col].astype("Int64")

    if "edad" in df.columns:
        df["edad"] = pd.to_numeric(df["edad"], errors="coerce")

    for col in ["sexo","tip_cas","tip_ss"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    return df

In [5]:
def vistazo(df):
    print("="*60)
    print(f"📊 La base de datos tiene {df.shape[0]:,} registros y {df.shape[1]} variables")
    print("="*60)

    # Información general del DataFrame
    print("\n🔎 Información del DataFrame:\n")
    df.info()

    print("="*60)


In [6]:
def eliminar_por_faltantes(df, faltantes_df, umbral=70):
    """
    Elimina columnas de un DataFrame según porcentaje de faltantes.
    
    Parámetros:
    - df: DataFrame original
    - faltantes_df: DataFrame con columnas ['faltantes', 'porcentaje'] 
                    y variables en el índice o en columna 'variable'
    - umbral: porcentaje a partir del cual se eliminan variables
    
    Retorna:
    - DataFrame limpio
    """
    
    # Detectar si las variables están en columna o en índice
    if "variable" in faltantes_df.columns:
        vars_eliminar = faltantes_df.loc[
            faltantes_df["porcentaje"] > umbral, "variable"
        ].tolist()
    else:
        vars_eliminar = faltantes_df[
            faltantes_df["porcentaje"] > umbral
        ].index.tolist()
    
    print(f"Eliminando {len(vars_eliminar)} variables")
    
    return df.drop(columns=vars_eliminar, errors="ignore")

In [7]:
def tablas_contingencia(df):
    """
    Genera una tabla de contingencia para cada variable categórica del dataframe.
    Muestra categorías, frecuencia absoluta y frecuencia relativa.
    """
    categoricas = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    for col in categoricas:
        print(f"\n{'='*60}")
        print(f"Variable: {col.upper()}")
        print(f"{'='*60}")
        
        tabla = (
            df[col]
            .value_counts(dropna=False)
            .rename_axis('Categoría')
            .reset_index(name='Frecuencia Absoluta')
        )
        
        tabla['Frecuencia Relativa (%)'] = (
            tabla['Frecuencia Absoluta'] / len(df) * 100
        ).round(2)
        
        print(tabla.to_string(index=False))
        print(f"\nTotal registros: {len(df):,} | Categorías únicas: {df[col].nunique(dropna=False)}")

## **1.3. Cargue de Datos:**

En esta sección se cargarán los datos epidemiológicos extraídos del portal oficial de Sivigila, correspondiente al sistema de vigilancia en salud pública del Instituto Nacional de Salud de Colombia: [SIVIGILA](https://portalsivigila.ins.gov.co/Paginas/Buscador.aspx?utm_source=chatgpt.com). La información utilizada comprende el periodo entre 2012 y 2024. Adicionalmente, se incorporaron los registros correspondientes al año 2011 con el propósito de construir variables temporales rezagadas (lags), las cuales requieren observaciones previas al inicio del periodo de análisis principal.


### **1.3.1. Datos 2011**

In [11]:
df_2011 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2011_210.xls")
pd.set_option('display.max_columns',None)
df_2011.head()

,Partición,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,Datos_2011_210,4265187,210,2011-02-11,6,2011,2001300591,1,4,1,NaN,NaN,F,170,20,13,1,9997,S,EPS033,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,20,13,20,20013,2011-02-11,2011-02-09,2,1,2011-02-11,1,NaN,0,NaN,NaN,NaN,2011-02-16,2011-02-16,NaN,NaN,NaN,0,2080598,1,2,Probable,ESE HOSPITAL AGUSTIN CODAZZI,COLOMBIA,DENGUE,CESAR,AGUSTIN CODAZZI,NaN,CESAR,AGUSTIN CODAZZI,CESAR,AGUSTIN CODAZZI
1,Datos_2011_210,4265188,210,2011-10-15,41,2011,7683401477,1,31,1,NaN,NaN,F,170,76,834,1,9999,C,EPS013,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,76,834,76,76834,2011-10-15,2011-10-12,3,1,2011-10-15,1,NaN,0,1980-08-04,NaN,NaN,2011-10-19,2011-10-19,NaN,NaN,NaN,1,2080599,1,3,Confirmado por laboratorio,CLINICA MEDICO QUIRURGICA ALVERNIA LTDA,COLOMBIA,DENGUE,VALLE,TULUA,NaN,VALLE,TULUA,VALLE,TULUA
2,Datos_2011_210,4265189,210,2011-07-11,27,2011,2300100297,0,47,1,NaN,NaN,M,170,23,1,1,5320,S,CCF002,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,23,1,23,23001,2011-07-11,2011-07-03,3,1,2011-07-11,1,NaN,0,1964-03-14,NaN,NaN,2011-07-18,2011-07-18,NaN,NaN,NaN,1,2080600,1,3,Confirmado por laboratorio,CLINICA ZAYMA LTDA,COLOMBIA,DENGUE,CORDOBA,MONTERIA,NaN,CORDOBA,MONTERIA,CORDOBA,MONTERIA
3,Datos_2011_210,4260779,210,2011-09-25,38,2011,7326800794,1,5,2,NaN,NaN,M,170,73,585,3,9998,S,EPS033,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,73,585,73,73268,2011-09-25,2011-09-24,2,2,NaN,1,NaN,0,2011-04-10,NaN,NaN,2011-09-28,2011-09-28,NaN,NaN,NaN,0,2080172,1,2,Probable,HOSPITAL SAN RAFAEL EMPRESA SOCIAL DEL ESTADO,COLOMBIA,DENGUE,TOLIMA,PURIFICACION,NaN,TOLIMA,PURIFICACION,TOLIMA,ESPINAL
4,Datos_2011_210,4270131,210,2011-01-15,1,2011,2736100180,1,39,1,NaN,NaN,M,170,27,745,1,6111,N,NaN,5,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,27,745,27,27361,2011-01-15,2011-01-08,3,2,NaN,1,NaN,0,1972-02-06,NaN,NaN,2011-01-17,2011-01-17,NaN,NaN,NaN,1,2081264,1,3,Confirmado por laboratorio,CENTRO MEDICO CUBIS,COLOMBIA,DENGUE,CHOCO,SIPI,NaN,CHOCO,SIPI,CHOCO,ITSMINA


In [12]:
vistazo(df_2011)

📊 La base de datos tiene 29,386 registros y 74 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29386 entries, 0 to 29385
Data columns (total 74 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Partición                  29386 non-null  object 
 1   CONSECUTIVE                29386 non-null  int64  
 2   COD_EVE                    29386 non-null  int64  
 3   FEC_NOT                    29386 non-null  object 
 4   SEMANA                     29386 non-null  int64  
 5   ANO                        29386 non-null  int64  
 6   COD_PRE                    29386 non-null  int64  
 7   COD_SUB                    29386 non-null  int64  
 8   EDAD                       29386 non-null  int64  
 9   UNI_MED                    29386 non-null  int64  
 10  nacionalidad               0 non-null      float64
 11  nombre_nacionalidad        0 non-null      float64
 12  SEXO          

In [ ]:
faltantes_2011 = Resumen_Faltantes(df_2011)
faltantes_2011

,faltantes,porcentaje
nacionalidad,29386,100.000
nombre_nacionalidad,29386,100.000
nom_grupo,29386,100.000
estrato,29386,100.000
sem_ges,29386,100.000
fuente,29386,100.000
COD_PAIS_R,29386,100.000
FM_UNIDAD,29386,100.000
FM_GRADO,29386,100.000
Pais_residencia,29386,100.000


### **1.3.2. Datos 2012**

In [ ]:
df_2012 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2012_210.xls")
pd.set_option('display.max_columns',None)
df_2012.head()

,Partición,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,Datos_2012_210,735411,210,2012-02-04,5,2012,7367100704,1,14,1,NaN,NaN,M,170,73,671,3,9997,S,CCF037,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,73,671,73,73671,2012-02-03,2012-01-31,2,2,NaN,1,NaN,7,1997-09-05,NaN,NaN,2013-04-05,2012-09-18,NaN,NaN,NaN,0,2577719,1,2,Probable,HOSPITAL SAN CARLOS,COLOMBIA,DENGUE,TOLIMA,SALDAÑA,NaN,TOLIMA,SALDAÑA,TOLIMA,SALDAÑA
1,Datos_2012_210,735412,210,2012-11-17,46,2012,5481001073,1,15,1,NaN,NaN,F,170,54,810,2,9999,S,999999,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,54,810,54,54810,2012-11-17,2012-11-15,2,2,NaN,1,NaN,0,1997-09-05,NaN,NaN,2013-04-05,2012-11-19,NaN,NaN,NaN,0,2577720,1,2,Probable,EMPRESA SOCIAL DEL ESTADO HOSPITAL REGIONAL NORTE,COLOMBIA,DENGUE,NORTE SANTANDER,TIBU,NaN,NORTE SANTANDER,TIBU,NORTE SANTANDER,TIBU
2,Datos_2012_210,735413,210,2012-04-05,14,2012,7631802027,0,14,1,NaN,NaN,M,170,76,318,1,9997,C,EPS001,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,76,318,76,76318,2012-04-05,2012-04-04,2,2,NaN,1,NaN,0,1997-09-05,NaN,NaN,2013-04-05,2012-04-10,NaN,NaN,NaN,0,2577721,1,2,Probable,HOSPITAL SAN ROQUE E.S.E. DE GUACARI,COLOMBIA,DENGUE,VALLE,GUACARI,NaN,VALLE,GUACARI,VALLE,GUACARI
3,Datos_2012_210,735414,210,2012-05-30,22,2012,4100100451,15,14,1,NaN,NaN,M,170,41,1,1,9997,S,CCF024,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,41,1,41,41001,2012-05-29,2012-05-29,2,2,NaN,1,NaN,3,1997-09-05,NaN,NaN,2013-04-05,2012-06-23,NaN,NaN,NaN,1,2577722,1,3,Confirmado por laboratorio,ESE CEO CENTRO DE SALUD SIETE DE AGOSTO,COLOMBIA,DENGUE,HUILA,NEIVA,NaN,HUILA,NEIVA,HUILA,NEIVA
4,Datos_2012_210,735415,210,2012-12-18,50,2012,6800103079,7,15,1,NaN,NaN,F,170,68,276,1,9997,C,EPS016,6,5,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,68,1,68,68001,2012-12-18,2012-12-14,2,2,NaN,1,NaN,0,1997-09-06,NaN,NaN,2013-04-05,2012-12-21,NaN,NaN,NaN,0,2577723,1,2,Probable,PUNTO VERDE COOMEVA,COLOMBIA,DENGUE,SANTANDER,FLORIDABLANCA,NaN,SANTANDER,BUCARAMANGA,SANTANDER,BUCARAMANGA


In [ ]:
vistazo(df_2012)

📊 La base de datos tiene 52,467 registros y 74 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52467 entries, 0 to 52466
Data columns (total 74 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Partición                  52467 non-null  object 
 1   CONSECUTIVE                52467 non-null  int64  
 2   COD_EVE                    52467 non-null  int64  
 3   FEC_NOT                    52467 non-null  object 
 4   SEMANA                     52467 non-null  int64  
 5   ANO                        52467 non-null  int64  
 6   COD_PRE                    52467 non-null  int64  
 7   COD_SUB                    52467 non-null  int64  
 8   EDAD                       52467 non-null  int64  
 9   UNI_MED                    52467 non-null  int64  
 10  nacionalidad               0 non-null      float64
 11  nombre_nacionalidad        0 non-null      float64
 12  SEXO          

In [ ]:
faltantes_2012 = Resumen_Faltantes(df_2012)
faltantes_2012

,faltantes,porcentaje
nacionalidad,52467,100.000
nombre_nacionalidad,52467,100.000
nom_grupo,52467,100.000
estrato,52467,100.000
sem_ges,52467,100.000
fuente,52467,100.000
COD_PAIS_R,52467,100.000
FM_UNIDAD,52467,100.000
FM_GRADO,52467,100.000
Pais_residencia,52467,100.000


### **1.3.3. Datos 2013**

In [ ]:
df_2013 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2013_210.xlsx")
pd.set_option('display.max_columns',None)
df_2013.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,540717,210,2013-12-28,52,2013,4443000168,0,70,1,NaN,NaN,M,170,5,660,1,9950,S,UT-004,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,5,660,44,44430,2013-04-05,2013-04-05,2,1,2013-04-06,1,NaN,3,1942-11-17,NaN,NaN,2014-03-06,2013-04-30,,NaN,NaN,1,150637,1,3,Confirmado por laboratorio,SOCIEDAD MEDICA CLINICA MAICAO,COLOMBIA,DENGUE,ANTIOQUIA,SAN LUIS,NaN,ANTIOQUIA,SAN LUIS,GUAJIRA,MAICAO
1,540718,210,2013-11-18,46,2013,4443000168,0,62,1,NaN,NaN,M,170,5,660,1,9950,S,UT-004,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,5,660,44,44430,2013-02-10,2013-02-07,2,2,NaN,1,NaN,0,1950-02-12,NaN,NaN,2014-03-06,2013-02-11,,NaN,NaN,0,150638,1,2,Probable,SOCIEDAD MEDICA CLINICA MAICAO,COLOMBIA,DENGUE,ANTIOQUIA,SAN LUIS,NaN,ANTIOQUIA,SAN LUIS,GUAJIRA,MAICAO
2,540719,210,2013-03-19,9,2013,4443000168,0,78,1,NaN,NaN,M,170,8,1,1,9950,S,ESS024,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,8,1,44,44430,2013-12-25,2013-12-22,2,2,NaN,1,NaN,0,1935-04-12,NaN,NaN,2014-03-06,2013-12-30,,NaN,NaN,0,150639,1,2,Probable,SOCIEDAD MEDICA CLINICA MAICAO,COLOMBIA,DENGUE,ATLANTICO,BARRANQUILLA,NaN,ATLANTICO,BARRANQUILLA,GUAJIRA,MAICAO
3,540720,210,2013-11-25,47,2013,4443000168,0,56,1,NaN,NaN,M,170,5,142,3,6111,S,UT-004,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,5,142,44,44430,2013-08-07,2013-07-31,3,1,2013-08-07,1,NaN,0,1957-03-29,NaN,NaN,2014-03-06,2013-08-22,,NaN,NaN,1,150640,1,3,Confirmado por laboratorio,SOCIEDAD MEDICA CLINICA MAICAO,COLOMBIA,DENGUE,ANTIOQUIA,CARACOLI,NaN,ANTIOQUIA,CARACOLI,GUAJIRA,MAICAO
4,540721,210,2013-09-12,37,2013,4407800307,0,51,1,NaN,NaN,M,170,5,893,1,9999,C,EPS013,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,68,81,44,44078,2013-05-11,2013-05-07,2,2,NaN,1,NaN,0,1961-10-29,NaN,NaN,2014-03-06,2013-05-13,,NaN,NaN,0,150641,1,2,Probable,ESE HOSPITAL NUESTRA SEÑORA DEL PILAR,COLOMBIA,DENGUE,ANTIOQUIA,YONDO (CASABE),NaN,SANTANDER,BARRANCABERMEJA,GUAJIRA,BARRANCAS


In [ ]:
vistazo(df_2013)

📊 La base de datos tiene 122,441 registros y 73 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122441 entries, 0 to 122440
Data columns (total 73 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CONSECUTIVE                122441 non-null  int64  
 1   COD_EVE                    122441 non-null  int64  
 2   FEC_NOT                    122441 non-null  object 
 3   SEMANA                     122441 non-null  int64  
 4   ANO                        122441 non-null  int64  
 5   COD_PRE                    122441 non-null  int64  
 6   COD_SUB                    122441 non-null  int64  
 7   EDAD                       122441 non-null  int64  
 8   UNI_MED                    122441 non-null  int64  
 9   nacionalidad               0 non-null       float64
 10  nombre_nacionalidad        0 non-null       float64
 11  SEXO                       122441 non-null  object 
 1

In [ ]:
faltantes_2013 = Resumen_Faltantes(df_2013)
faltantes_2013

,faltantes,porcentaje
nacionalidad,122441,100.000
nombre_nacionalidad,122441,100.000
GRU_POB,122441,100.000
fuente,122441,100.000
nom_grupo,122441,100.000
estrato,122441,100.000
sem_ges,122441,100.000
Pais_residencia,122441,100.000
COD_PAIS_R,122441,100.000
FEC_DEF,122431,99.992


### **1.3.4. Datos 2014**

In [ ]:
df_2014 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2014_210.xlsx")
pd.set_option('display.max_columns',None)
df_2014.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,1722369,210,2014-12-07,49,2014,7000101049,1,10,1,NaN,NaN,M,170,70,1,1,9997,S,ESS024,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,70,1,70,70001,2014-12-07,2014-12-04,2,1,2014-12-07,1,NaN,3,2004-03-28,NaN,NaN,2015-03-04,2014-12-22,NaN,NaN,NaN,1,67762,1,3,Confirmado por laboratorio,CLINICA PEDIATRICA NIÑO JESUS LTDA,COLOMBIA,DENGUE,SUCRE,SINCELEJO,NaN,SUCRE,SINCELEJO,SUCRE,SINCELEJO
1,1722370,210,2014-06-14,24,2014,1354900095,1,1,1,NaN,NaN,F,170,13,549,2,9999,S,ESS207,6,NaN,NaN,NaN,2,1,2,2,2,NaN,2,2,2,2,2,2,2,NaN,NaN,13,549,13,13549,2014-06-10,2014-06-09,3,1,2014-06-13,1,NaN,0,2013-01-09,NaN,NaN,2015-03-04,2014-06-17,NaN,NaN,NaN,1,67763,1,3,Confirmado por laboratorio,ESE HOSPITAL SAN NICOLAS DE TOLENTINO,COLOMBIA,DENGUE,BOLIVAR,PINILLOS,NaN,BOLIVAR,PINILLOS,BOLIVAR,PINILLOS
2,1722371,210,2014-02-21,8,2014,1343000492,1,6,2,NaN,NaN,F,170,13,549,1,9999,S,ESS207,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,13,549,13,13430,2014-02-21,2014-02-19,3,1,2014-02-21,1,NaN,3,2013-08-14,NaN,NaN,2015-03-04,2014-05-22,NaN,NaN,NaN,1,67764,1,3,Confirmado por laboratorio,ESE HOSPITAL LA DIVINA MISERICORDIA,COLOMBIA,DENGUE,BOLIVAR,PINILLOS,NaN,BOLIVAR,PINILLOS,BOLIVAR,MAGANGUE
3,1722372,210,2014-11-06,44,2014,4724500605,1,6,1,NaN,NaN,M,170,13,667,2,9998,S,ESS133,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,13,667,47,47245,2014-10-31,2014-10-29,3,1,2014-11-03,1,NaN,7,2008-01-10,NaN,NaN,2015-03-04,2015-03-03,NaN,NaN,NaN,1,67765,1,3,Confirmado por laboratorio,PREVENCION Y SALUD IPS LIMITADA,COLOMBIA,DENGUE,BOLIVAR,SAN MARTIN DE LOBA,NaN,BOLIVAR,SAN MARTIN DE LOBA,MAGDALENA,EL BANCO
4,1722373,210,2014-12-04,49,2014,2530700004,1,47,1,NaN,NaN,M,170,25,307,1,5149,C,EPS005,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,25,307,25,25307,2014-12-04,2014-12-01,2,2,NaN,1,NaN,0,1967-04-17,NaN,NaN,2015-03-04,2014-12-05,NaN,NaN,NaN,0,67766,1,2,Probable,SOCIEDAD DE ESPECIALISTAS DE GIRARDOT,COLOMBIA,DENGUE,CUNDINAMARCA,GIRARDOT,NaN,CUNDINAMARCA,GIRARDOT,CUNDINAMARCA,GIRARDOT


In [ ]:
vistazo(df_2014)

📊 La base de datos tiene 105,356 registros y 73 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105356 entries, 0 to 105355
Data columns (total 73 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CONSECUTIVE                105356 non-null  int64  
 1   COD_EVE                    105356 non-null  int64  
 2   FEC_NOT                    105356 non-null  object 
 3   SEMANA                     105356 non-null  int64  
 4   ANO                        105356 non-null  int64  
 5   COD_PRE                    105356 non-null  int64  
 6   COD_SUB                    105356 non-null  int64  
 7   EDAD                       105356 non-null  int64  
 8   UNI_MED                    105356 non-null  int64  
 9   nacionalidad               0 non-null       float64
 10  nombre_nacionalidad        0 non-null       float64
 11  SEXO                       105356 non-null  object 
 1

In [ ]:
faltantes_2014 = Resumen_Faltantes(df_2014)
faltantes_2014

,faltantes,porcentaje
nacionalidad,105356,100.000
nombre_nacionalidad,105356,100.000
GRU_POB,105356,100.000
Pais_residencia,105356,100.000
nom_grupo,105356,100.000
estrato,105356,100.000
sem_ges,105356,100.000
fuente,105356,100.000
COD_PAIS_R,105356,100.000
CER_DEF,105356,100.000


### **1.3.5. Datos 2015**

In [ ]:
df_2015 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2015_210.xlsx")
df_2015.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,963907,210,2015-05-16,19,2015,6800101628,3,56,1,NaN,NaN,F,170,68,307,1,9996,C,EPS002,2,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,68,307,68,68001,2015-05-16,2015-05-12,2,2,NaN,1,NaN,0,1959-04-18,NaN,NaN,2016-03-03,2015-05-23,NaN,NaN,NaN,0,1618629,1,2,Probable,SALUD TOTAL EPS UUBC,COLOMBIA,DENGUE,SANTANDER,GIRON,NaN,SANTANDER,GIRON,SANTANDER,BUCARAMANGA
1,963908,210,2015-02-27,7,2015,1945508017,1,55,1,NaN,NaN,M,170,19,455,1,9999,C,EPS016,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,19,455,19,19455,2015-02-24,2015-02-19,3,2,NaN,1,NaN,0,1959-04-18,NaN,NaN,2016-03-03,2015-03-02,NaN,NaN,NaN,1,1618630,1,3,Confirmado por laboratorio,IPS CLINICA SALUD FLORIDA SEDE MIRANDA,COLOMBIA,DENGUE,CAUCA,MIRANDA,NaN,CAUCA,MIRANDA,CAUCA,MIRANDA
2,963909,210,2015-04-14,15,2015,875800161,4,56,1,NaN,NaN,M,170,8,758,1,9950,S,ESS076,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,8,758,8,8758,2015-04-14,2015-04-14,2,2,NaN,1,NaN,3,1959-04-19,NaN,NaN,2016-03-03,2015-05-20,NaN,NaN,NaN,1,1618631,1,3,Confirmado por laboratorio,EMPRESA SOCIAL DEL ESTADO HMI 13 DE JUNIO,COLOMBIA,DENGUE,ATLANTICO,SOLEDAD,NaN,ATLANTICO,SOLEDAD,ATLANTICO,SOLEDAD
3,963910,210,2015-07-07,3,2015,7300101996,1,56,1,NaN,NaN,M,170,73,1,1,9115,C,EPS003,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,73,1,73,73001,2015-01-23,2015-01-21,2,2,NaN,1,NaN,0,1959-04-19,NaN,NaN,2016-03-03,2015-07-28,NaN,NaN,NaN,0,1618632,1,2,Probable,CENTRAL DE URGENCIAS ESPINAL CORPORACION IPS S...,COLOMBIA,DENGUE,TOLIMA,IBAGUE,NaN,TOLIMA,IBAGUE,TOLIMA,IBAGUE
4,963911,210,2015-03-05,9,2015,2559600352,2,55,1,NaN,NaN,M,170,25,596,2,6111,S,EPS022,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,25,596,25,25596,2015-03-05,2015-03-04,2,2,NaN,1,NaN,0,1959-04-19,NaN,NaN,2016-03-03,2015-03-07,NaN,NaN,NaN,0,1618633,1,2,Probable,CENTRO DE SALUD QUIPILE,COLOMBIA,DENGUE,CUNDINAMARCA,QUIPILE,NaN,CUNDINAMARCA,QUIPILE,CUNDINAMARCA,QUIPILE


In [ ]:
vistazo(df_2015)

📊 La base de datos tiene 95,023 registros y 73 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95023 entries, 0 to 95022
Data columns (total 73 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CONSECUTIVE                95023 non-null  int64  
 1   COD_EVE                    95023 non-null  int64  
 2   FEC_NOT                    95023 non-null  object 
 3   SEMANA                     95023 non-null  int64  
 4   ANO                        95023 non-null  int64  
 5   COD_PRE                    95023 non-null  int64  
 6   COD_SUB                    95023 non-null  int64  
 7   EDAD                       95023 non-null  int64  
 8   UNI_MED                    95023 non-null  int64  
 9   nacionalidad               0 non-null      float64
 10  nombre_nacionalidad        0 non-null      float64
 11  SEXO                       95023 non-null  object 
 12  COD_PAIS_O    

In [ ]:
faltantes_2015 = Resumen_Faltantes(df_2015)
faltantes_2015

,faltantes,porcentaje
nacionalidad,95023,100.000
nombre_nacionalidad,95023,100.000
nom_grupo,95023,100.000
COD_PAIS_R,95023,100.000
estrato,95023,100.000
sem_ges,95023,100.000
fuente,95023,100.000
CER_DEF,95023,100.000
Pais_residencia,95023,100.000
FEC_DEF,95023,100.000


### **1.3.6. Datos 2016**

In [ ]:
df_2016 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2016_210.xlsx")
df_2016.head()

,COD_EVE,CONSECUTIVE,COD_EVE.1,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,210,2169060,210,2016-11-04,42,2016,7600100037,6,63,1,NaN,NaN,F,170,76,1,1,9999,C,EPS018,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,76,1,76,76001,2016-11-04,2016-10-22,2,2,NaN,1,NaN,0,1953-01-30,NaN,NaN,2017-03-08,2016-11-08,NaN,NaN,NaN,0,21955,1,2,Probable,COMFANDI IPS CALIPSO,COLOMBIA,DENGUE,VALLE,CALI,NaN,VALLE,CALI,VALLE,CALI
1,210,7724,210,2016-05-07,18,2016,1953200012,1,39,1,NaN,NaN,F,170,19,532,2,9996,S,ESS062,5,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,19,532,19,19532,2016-05-07,2016-05-01,5,2,NaN,1,NaN,0,1976-11-11,NaN,NaN,2017-03-08,2016-05-07,NaN,NaN,NaN,1,69947,1,5,Confirmado por Nexo Epidemiológico,ESE HOSPITAL NIVEL I EL BORDO,COLOMBIA,DENGUE,CAUCA,PATIA (EL BORDO),NaN,CAUCA,PATIA (EL BORDO),CAUCA,PATIA (EL BORDO)
2,210,7802,210,2016-07-30,29,2016,1100104035,3,39,1,NaN,NaN,F,170,68,0,2,2419,C,EPS001,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,11,1,11,11001,2016-07-26,2016-07-21,3,2,NaN,1,NaN,0,1977-05-30,NaN,NaN,2017-03-08,2016-08-02,NaN,NaN,NaN,1,69948,1,3,Confirmado por laboratorio,CENTRO MEDICO COLMEDICA MEDICINA PREPAGADA SAN...,COLOMBIA,DENGUE,SANTANDER,* SANTANDER. MUNICIPIO DESCONOCIDO,NaN,BOGOTA,BOGOTA,BOGOTA,BOGOTA
3,210,7804,210,2016-12-02,48,2016,1568600845,1,40,1,NaN,NaN,F,170,15,686,3,9996,S,ESS133,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,15,686,15,15686,2016-11-30,2016-11-30,2,2,NaN,1,NaN,3,1976-07-17,NaN,NaN,2017-03-08,2017-01-24,NaN,NaN,NaN,1,69949,1,3,Confirmado por laboratorio,ESE CENTRO DE SALUD SANTANA,COLOMBIA,DENGUE,BOYACA,SANTANA,NaN,BOYACA,SANTANA,BOYACA,SANTANA
4,210,7958,210,2016-11-26,45,2016,1100108171,36,38,1,NaN,NaN,F,170,73,449,2,8290,C,EPS017,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,11,1,11,11001,2016-11-26,2016-11-11,3,2,NaN,1,NaN,0,1978-01-22,NaN,NaN,2017-03-08,2016-11-29,NaN,NaN,NaN,1,70205,1,3,Confirmado por laboratorio,CENTRO MEDICO COLSUBSIDIO QUIROGA - RUU,COLOMBIA,DENGUE,TOLIMA,MELGAR,NaN,BOGOTA,BOGOTA,BOGOTA,BOGOTA


In [ ]:
vistazo(df_2016)

📊 La base de datos tiene 100,117 registros y 74 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100117 entries, 0 to 100116
Data columns (total 74 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   COD_EVE                    100117 non-null  int64  
 1   CONSECUTIVE                100117 non-null  int64  
 2   COD_EVE.1                  100117 non-null  int64  
 3   FEC_NOT                    100117 non-null  object 
 4   SEMANA                     100117 non-null  int64  
 5   ANO                        100117 non-null  int64  
 6   COD_PRE                    100117 non-null  int64  
 7   COD_SUB                    100117 non-null  int64  
 8   EDAD                       100117 non-null  int64  
 9   UNI_MED                    100117 non-null  int64  
 10  nacionalidad               0 non-null       float64
 11  nombre_nacionalidad        0 non-null       float64
 1

In [ ]:
faltantes_2016 = Resumen_Faltantes(df_2016)
faltantes_2016

,faltantes,porcentaje
nacionalidad,100117,100.000
nombre_nacionalidad,100117,100.000
GRU_POB,100117,100.000
nom_grupo,100117,100.000
COD_PAIS_R,100117,100.000
estrato,100117,100.000
sem_ges,100117,100.000
fuente,100117,100.000
Pais_residencia,100117,100.000
FEC_DEF,100115,99.998


### **1.3.7. Datos 2017**

In [ ]:
df_2017 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2017_210.xls")
df_2017.head(2)

,Partición,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,Datos_2017_210,5444398,210,2017-07-18,28,2017,2300100553,61,5,1,NaN,NaN,F,170,23,1,1,9997,S,ESS207,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,23,1,23,23001,2017-07-15,2017-07-12,2,2,NaN,1,NaN,0,NaN,NaN,NaN,2018-04-12,2017-07-18,NaN,NaN,NaN,0,4084,1,2,Probable,ESE VIDASINU HOSPITAL LA GLORIA,COLOMBIA,DENGUE,CORDOBA,MONTERIA,NaN,CORDOBA,MONTERIA,CORDOBA,MONTERIA
1,Datos_2017_210,5445137,210,2017-12-14,49,2017,1100116338,83,18,1,NaN,NaN,M,170,25,488,2,9997,S,ESS133,6,NaN,NaN,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,15,47,11,11001,2017-12-08,2017-12-04,3,1,2017-12-08,1,NaN,0,NaN,NaN,NaN,2018-04-12,2017-12-14,NaN,NaN,NaN,1,2920,1,3,Confirmado por laboratorio,HOSPITAL MILITAR CENTRAL CHAPINERO,COLOMBIA,DENGUE,CUNDINAMARCA,NILO,NaN,BOYACA,AQUITANIA,BOGOTA,BOGOTA


In [ ]:
vistazo(df_2017)

📊 La base de datos tiene 25,048 registros y 74 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25048 entries, 0 to 25047
Data columns (total 74 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Partición                  25048 non-null  object 
 1   CONSECUTIVE                25048 non-null  int64  
 2   COD_EVE                    25048 non-null  int64  
 3   FEC_NOT                    25048 non-null  object 
 4   SEMANA                     25048 non-null  int64  
 5   ANO                        25048 non-null  int64  
 6   COD_PRE                    25048 non-null  int64  
 7   COD_SUB                    25048 non-null  int64  
 8   EDAD                       25048 non-null  int64  
 9   UNI_MED                    25048 non-null  int64  
 10  nacionalidad               0 non-null      float64
 11  nombre_nacionalidad        0 non-null      float64
 12  SEXO          

In [ ]:
faltantes_2017 = Resumen_Faltantes(df_2017)
faltantes_2017

,faltantes,porcentaje
nacionalidad,25048,100.000
nombre_nacionalidad,25048,100.000
GRU_POB,25048,100.000
Pais_residencia,25048,100.000
nom_grupo,25048,100.000
estrato,25048,100.000
sem_ges,25048,100.000
fuente,25048,100.000
COD_PAIS_R,25048,100.000
FECHA_NTO,25048,100.000


### **1.3.8. Datos 2018**

In [ ]:
df_2018 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2018_210.xls")
df_2018.head()

,Partición,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,Datos_2018_210,5581586,210,2018-02-22,7,2018,4443000687,1,14,1,,...,F,170,44,430,1,9997,S,CCF023,6,NaN,...,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,44,430,44,44430,2018-02-22,2018-02-16,3,2,NaN,1,NaN,0,2003-09-16,NaN,NaN,2019-05-08,2018-02-27,NaN,NaN,NaN,1,42383,1,3,Confirmado por laboratorio,IPSI AYUULEEPALA WAYUU,COLOMBIA,DENGUE,GUAJIRA,MAICAO,NaN,GUAJIRA,MAICAO,GUAJIRA,MAICAO
1,Datos_2018_210,5581587,210,2018-01-19,3,2018,4443000706,1,16,1,,...,M,170,44,430,1,9997,S,RES005,6,NaN,...,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,44,430,44,44430,2018-01-18,2018-01-14,3,1,2018-01-19,1,NaN,0,2001-10-10,NaN,NaN,2019-05-08,2018-01-23,NaN,NaN,NaN,1,42384,1,3,Confirmado por laboratorio,CLINICA DE ESPECIALISTAS GUAJIRA LTDA,COLOMBIA,DENGUE,GUAJIRA,MAICAO,NaN,GUAJIRA,MAICAO,GUAJIRA,MAICAO
2,Datos_2018_210,5581588,210,2018-11-20,46,2018,5059000885,1,15,1,,...,F,170,50,590,1,9997,S,EPSS34,6,NaN,...,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,50,590,50,50590,2018-11-20,2018-11-15,2,2,NaN,1,NaN,0,2003-08-08,NaN,NaN,2019-05-08,2018-11-23,NaN,NaN,NaN,0,42385,1,2,Probable,ESE HOSPITAL NIVEL 1 PUERTO RICO,COLOMBIA,DENGUE,META,PUERTO RICO,NaN,META,PUERTO RICO,META,PUERTO RICO
3,Datos_2018_210,5581589,210,2018-06-03,22,2018,5000100529,1,15,1,,...,M,170,50,1,1,9997,S,EPSS34,6,NaN,...,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,50,1,50,50001,2018-06-02,2018-06-01,2,2,NaN,1,NaN,3,2003-04-09,NaN,NaN,2019-05-08,2018-06-13,NaN,NaN,NaN,1,42386,1,3,Confirmado por laboratorio,HOSPITAL DEPARTAMENTAL DE VILLAVICENCIO,COLOMBIA,DENGUE,META,VILLAVICENCIO,NaN,META,VILLAVICENCIO,META,VILLAVICENCIO
4,Datos_2018_210,5581590,210,2018-09-30,39,2018,5031300522,1,16,1,,...,M,170,50,313,2,9997,S,EPSS34,6,NaN,...,NaN,2,2,2,2,2,NaN,2,2,2,2,2,2,1,NaN,NaN,50,313,50,50313,2018-09-30,2018-09-28,2,2,NaN,1,NaN,0,2002-06-24,NaN,NaN,2019-05-08,2018-10-05,NaN,NaN,NaN,0,42387,1,2,Probable,HOSPITAL DEPARTAMENTAL DE GRANADA ESE,COLOMBIA,DENGUE,META,GRANADA,NaN,META,GRANADA,META,GRANADA


In [ ]:
vistazo(df_2018)

📊 La base de datos tiene 43,652 registros y 74 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43652 entries, 0 to 43651
Data columns (total 74 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Partición                  43652 non-null  object 
 1   CONSECUTIVE                43652 non-null  int64  
 2   COD_EVE                    43652 non-null  int64  
 3   FEC_NOT                    43652 non-null  object 
 4   SEMANA                     43652 non-null  int64  
 5   ANO                        43652 non-null  int64  
 6   COD_PRE                    43652 non-null  int64  
 7   COD_SUB                    43652 non-null  int64  
 8   EDAD                       43652 non-null  int64  
 9   UNI_MED                    43652 non-null  int64  
 10  nacionalidad               43652 non-null  object 
 11  nombre_nacionalidad        43652 non-null  object 
 12  SEXO          

In [ ]:
faltantes_2018 = Resumen_Faltantes(df_2018)
faltantes_2018

,faltantes,porcentaje
GRU_POB,43652,100.000
estrato,43652,100.000
sem_ges,43652,100.000
COD_PAIS_R,43652,100.000
fuente,43652,100.000
CER_DEF,43652,100.000
FEC_DEF,43652,100.000
CBMTE,43652,100.000
Pais_residencia,43652,100.000
FM_UNIDAD,43286,99.162


### **1.3.9. Datos 2019**

In [ ]:
df_2019 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2019_210.xlsx")
df_2019.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,6481307,210,2019-11-09,45,2019,4746000051,2,7,1,170,COLOMBIA ...,M,170,47,460,2,9997,S,ESS207,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,NaN,47,460,47,47460,2019-11-09,2019-11-09,3,1,2019-11-09,1,NaN,7,2012-02-05,NaN,NaN,2020-06-17,2020-06-17,,NaN,NaN,1,4908,1,3,Confirmado por laboratorio,CENTRO DE SALUD LOS ANDES,COLOMBIA,DENGUE,MAGDALENA,NUEVA GRANADA,NaN,MAGDALENA,NUEVA GRANADA,MAGDALENA,NUEVA GRANADA
1,6481308,210,2019-05-08,18,2019,4755500106,1,7,1,170,COLOMBIA ...,M,170,47,460,1,9997,S,ESS207,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,NaN,47,460,47,47555,2019-05-08,2019-05-04,3,1,2019-05-08,1,NaN,0,2012-03-13,NaN,NaN,2020-06-17,2019-05-13,,NaN,NaN,1,4920,1,3,Confirmado por laboratorio,ESE FRAY LUIS DE LEON,COLOMBIA,DENGUE,MAGDALENA,NUEVA GRANADA,NaN,MAGDALENA,NUEVA GRANADA,MAGDALENA,PLATO
2,6481309,210,2019-01-22,3,2019,4755500106,1,6,1,,...,F,170,47,460,1,9996,S,ESS207,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,NaN,47,460,47,47555,2019-01-22,2019-01-17,3,2,NaN,1,NaN,0,2012-04-24,NaN,NaN,2020-06-17,2019-01-28,,NaN,NaN,1,4931,1,3,Confirmado por laboratorio,ESE FRAY LUIS DE LEON,COLOMBIA,DENGUE,MAGDALENA,NUEVA GRANADA,NaN,MAGDALENA,NUEVA GRANADA,MAGDALENA,PLATO
3,6481310,210,2019-02-15,7,2019,4746000051,1,6,1,,...,F,170,47,460,1,9999,S,ESS024,6,NaN,...,,2,2,2,2,2,,2,2,2,2,2,2,1,1,NaN,47,460,47,47460,2019-02-15,2019-02-15,3,1,2019-02-15,1,NaN,0,2012-05-10,NaN,NaN,2020-06-17,2019-02-15,,NaN,NaN,1,4943,1,3,Confirmado por laboratorio,ESE HOSPITAL LOCAL DE NUEVA GRANADA,COLOMBIA,DENGUE,MAGDALENA,NUEVA GRANADA,NaN,MAGDALENA,NUEVA GRANADA,MAGDALENA,NUEVA GRANADA
4,6481311,210,2019-06-25,25,2019,4755500106,1,25,1,170,COLOMBIA ...,M,170,47,460,3,9622,S,ESS024,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,NaN,47,460,47,47555,2019-06-25,2019-06-22,3,1,2019-06-25,1,NaN,0,1994-01-29,NaN,NaN,2020-06-17,2019-07-01,,NaN,NaN,1,4955,1,3,Confirmado por laboratorio,ESE FRAY LUIS DE LEON,COLOMBIA,DENGUE,MAGDALENA,NUEVA GRANADA,NaN,MAGDALENA,NUEVA GRANADA,MAGDALENA,PLATO


In [ ]:
vistazo(df_2019)

📊 La base de datos tiene 123,641 registros y 73 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123641 entries, 0 to 123640
Data columns (total 73 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CONSECUTIVE                123641 non-null  int64  
 1   COD_EVE                    123641 non-null  int64  
 2   FEC_NOT                    123641 non-null  object 
 3   SEMANA                     123641 non-null  int64  
 4   ANO                        123641 non-null  int64  
 5   COD_PRE                    123641 non-null  int64  
 6   COD_SUB                    123641 non-null  int64  
 7   EDAD                       123641 non-null  int64  
 8   UNI_MED                    123641 non-null  int64  
 9   nacionalidad               123641 non-null  object 
 10  nombre_nacionalidad        123641 non-null  object 
 11  SEXO                       123641 non-null  object 
 1

In [ ]:
faltantes_2019 = Resumen_Faltantes(df_2019)
faltantes_2019

,faltantes,porcentaje
GRU_POB,123641,100.000
COD_PAIS_R,123641,100.000
Pais_residencia,123641,100.000
FEC_DEF,123640,99.999
CBMTE,123640,99.999
CER_DEF,123640,99.999
FM_GRADO,121769,98.486
FM_UNIDAD,121769,98.486
FEC_HOS,63805,51.605
COD_ASE,3948,3.193


### **1.3.10. Datos 2020**

In [ ]:
df_2020 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2020_210.xlsx")
df_2020.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,7232769,210,2020-02-27,9,2020,6800104268,1,15,1,170,COLOMBIA ...,F,170,68,276,1,9997,C,EPS044,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,68,276,68,68001,2020-02-27,2020-02-23,2,1,2020-02-27,1,NaN,3,2004-08-26,NaN,NaN,2021-04-30,2020-03-16,,NaN,NaN,1,3155,1,3,Confirmado por laboratorio,CLINICA DE URGENCIAS BUCARAMANGA SAS,COLOMBIA,DENGUE,SANTANDER,FLORIDABLANCA,COLOMBIA,SANTANDER,FLORIDABLANCA,SANTANDER,BUCARAMANGA
1,7232770,210,2020-02-08,5,2020,5011000634,11,4,1,170,COLOMBIA ...,F,170,50,110,1,9999,C,EPS044,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,50,110,50,50110,2020-01-30,2020-01-30,2,1,2020-01-30,1,NaN,0,2015-10-13,NaN,NaN,2021-04-30,2020-02-09,,NaN,NaN,0,3167,1,2,Probable,CENTRO DE ATENCION BARRANCA DE UPIA,COLOMBIA,DENGUE,META,BARRANCA DE UPIA,COLOMBIA,META,BARRANCA DE UPIA,META,BARRANCA DE UPIA
2,7232771,210,2020-03-04,9,2020,7600103957,11,15,1,170,COLOMBIA ...,F,170,76,1,1,9999,S,ESS118,5,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,76,1,76,76001,2020-03-04,2020-02-26,2,2,NaN,1,NaN,0,2004-09-25,NaN,NaN,2021-04-30,2020-03-05,,NaN,NaN,0,3179,1,2,Probable,CENTRO DE SALUD MARROQUIN ESE ORIENTE,COLOMBIA,DENGUE,VALLE,CALI,COLOMBIA,VALLE,CALI,VALLE,CALI
3,7232772,210,2020-01-21,3,2020,2000100438,1,10,1,170,COLOMBIA ...,F,170,20,1,1,9997,I,NaN,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,20,1,20,20001,2020-01-21,2020-01-17,2,1,2020-01-21,1,NaN,3,2010-01-07,NaN,NaN,2021-04-30,2020-03-02,,NaN,NaN,1,3190,1,3,Confirmado por laboratorio,SOCIEDAD CLINICA VALLEDUPAR LTDA,COLOMBIA,DENGUE,CESAR,VALLEDUPAR,COLOMBIA,CESAR,VALLEDUPAR,CESAR,VALLEDUPAR
4,7232773,210,2020-05-19,20,2020,8500100001,3,10,1,170,COLOMBIA ...,F,170,85,230,1,9997,S,EPSS10,6,NaN,...,,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,85,230,85,85001,2020-05-17,2020-05-10,2,1,2020-05-17,1,NaN,3,2010-03-01,NaN,NaN,2021-04-30,2020-10-05,,NaN,NaN,1,3202,1,3,Confirmado por laboratorio,HOSPITAL DE YOPAL ESE NUEVA SEDE,COLOMBIA,DENGUE,CASANARE,OROCUE,COLOMBIA,CASANARE,OROCUE,CASANARE,YOPAL


In [ ]:
vistazo(df_2020)

📊 La base de datos tiene 76,419 registros y 73 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76419 entries, 0 to 76418
Data columns (total 73 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CONSECUTIVE                76419 non-null  int64  
 1   COD_EVE                    76419 non-null  int64  
 2   FEC_NOT                    76419 non-null  object 
 3   SEMANA                     76419 non-null  int64  
 4   ANO                        76419 non-null  int64  
 5   COD_PRE                    76419 non-null  int64  
 6   COD_SUB                    76419 non-null  int64  
 7   EDAD                       76419 non-null  int64  
 8   UNI_MED                    76419 non-null  int64  
 9   nacionalidad               76419 non-null  int64  
 10  nombre_nacionalidad        76419 non-null  object 
 11  SEXO                       76419 non-null  object 
 12  COD_PAIS_O    

In [ ]:
faltantes_2020 = Resumen_Faltantes(df_2020)
faltantes_2020

,faltantes,porcentaje
GRU_POB,76419,100.000
CBMTE,76419,100.000
FEC_DEF,76419,100.000
CER_DEF,76419,100.000
FM_UNIDAD,75088,98.258
FM_GRADO,75088,98.258
FEC_HOS,41924,54.861
COD_ASE,3729,4.880
FECHA_NTO,84,0.110


### **1.3.11. Datos 2021**

In [ ]:
df_2021 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2021_210.xls")
df_2021.head()

,Partición,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,Datos_2021_210,8564055,210,2021-05-22,16,2021,6800100431,1,17,1,170,COLOMBIA ...,F,170,68,307,1,9999,C,EPS016,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,68,307,68,68001,2021-04-24,2021-04-21,2,1,2021-04-24,1,NaN,3,2004-05-01,NaN,NaN,2021-09-08,2021-09-01,NaN,NaN,NaN,1,3288,1,3,Confirmado por laboratorio,CLINICA MATERNO INFANTEL SAN LUIS SA,COLOMBIA,DENGUE,SANTANDER,GIRON,COLOMBIA,SANTANDER,GIRON,SANTANDER,BUCARAMANGA
1,Datos_2021_210,8564056,210,2021-06-21,24,2021,8500104547,1,34,1,170,COLOMBIA ...,M,170,85,1,1,9622,C,EPS005,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,85,1,85,85001,2021-06-21,2021-06-16,2,2,NaN,1,NaN,0,1986-12-29,NaN,NaN,2021-06-30,2021-06-28,NaN,NaN,NaN,0,3289,1,2,Probable,LACOR YOPAL IPS SAS,COLOMBIA,DENGUE,CASANARE,YOPAL,COLOMBIA,CASANARE,YOPAL,CASANARE,YOPAL
2,Datos_2021_210,8564057,210,2021-08-01,29,2021,5283501489,1,34,1,170,COLOMBIA ...,M,170,52,835,1,9994,P,RES003,5,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,52,835,52,52835,2021-07-31,2021-07-24,3,1,2021-08-01,1,NaN,0,1987-02-27,NaN,NaN,2021-08-04,2021-08-02,NaN,NaN,NaN,1,3290,1,3,Confirmado por laboratorio,IPS PUENTE DEL MEDIO,COLOMBIA,DENGUE,NARIÑO,TUMACO,COLOMBIA,NARIÑO,TUMACO,NARIÑO,TUMACO
3,Datos_2021_210,8564058,210,2021-03-22,11,2021,1300181500,81,33,1,170,COLOMBIA ...,M,170,13,1,1,9994,P,RES003,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,13,1,13,13001,2021-03-22,2021-03-19,2,2,NaN,1,NaN,3,1987-04-26,NaN,NaN,2021-03-31,2021-03-29,4.0,1.400181e+09,064,1,3291,1,3,Confirmado por laboratorio,HOSPITAL NAVAL DE CARTAGENA,COLOMBIA,DENGUE,BOLIVAR,CARTAGENA,COLOMBIA,BOLIVAR,CARTAGENA,BOLIVAR,CARTAGENA
4,Datos_2021_210,8564059,210,2021-09-13,37,2021,6819000419,1,34,1,170,COLOMBIA ...,M,170,68,190,3,6112,S,EPSS37,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,68,190,68,68190,2021-09-13,2021-09-12,2,2,NaN,1,NaN,0,1987-05-28,NaN,NaN,2021-09-29,2021-09-20,NaN,NaN,NaN,0,3292,1,2,Probable,CLINICA SAN JOSE IPS LTDA CIMITARRA,COLOMBIA,DENGUE,SANTANDER,CIMITARRA,COLOMBIA,SANTANDER,CIMITARRA,SANTANDER,CIMITARRA


In [ ]:
vistazo(df_2021)

📊 La base de datos tiene 49,325 registros y 74 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49325 entries, 0 to 49324
Data columns (total 74 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Partición                  49325 non-null  object 
 1   CONSECUTIVE                49325 non-null  int64  
 2   COD_EVE                    49325 non-null  int64  
 3   FEC_NOT                    49325 non-null  object 
 4   SEMANA                     49325 non-null  int64  
 5   ANO                        49325 non-null  int64  
 6   COD_PRE                    49325 non-null  int64  
 7   COD_SUB                    49325 non-null  int64  
 8   EDAD                       49325 non-null  int64  
 9   UNI_MED                    49325 non-null  int64  
 10  nacionalidad               49325 non-null  int64  
 11  nombre_nacionalidad        49325 non-null  object 
 12  SEXO          

In [ ]:
faltantes_2021 = Resumen_Faltantes(df_2021)
faltantes_2021

,faltantes,porcentaje
GRU_POB,49325,100.000
FEC_DEF,49325,100.000
CBMTE,49325,100.000
CER_DEF,49325,100.000
FM_GRADO,48816,98.968
FM_FUERZA,48816,98.968
FM_UNIDAD,48816,98.968
FEC_HOS,24655,49.985
COD_ASE,2049,4.154
Municipio_residencia,4,0.008


### **1.3.12. Datos 2022**

In [ ]:
df_2022 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2022_210.xlsx")
df_2022.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,8915475,210,2022-09-30,39,2022,8541000080,1,19,1,170,COLOMBIA ...,F,170,85,410,1,9622,S,EPS025,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,85,410,85,85410,2022-09-30,2022-09-27,2,2,NaN,1,NaN,0,2003-10-01,NaN,NaN,2022-10-05,2022-10-03,NaN,NaN,NaN,0,32,1,2,Probable,ESE HOSPITAL LOCAL DE TAURAMENA,COLOMBIA,DENGUE,CASANARE,TAURAMENA,COLOMBIA,CASANARE,TAURAMENA,CASANARE,TAURAMENA
1,8915476,210,2022-02-07,5,2022,8501000190,1,18,1,170,COLOMBIA ...,F,170,85,10,1,9996,S,EPS025,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,85,10,85,85010,2022-02-07,2022-02-04,2,2,NaN,1,NaN,0,2003-11-03,NaN,NaN,2022-02-16,2022-02-09,NaN,NaN,NaN,0,33,1,2,Probable,HOSPITAL DE AGUAZUL JUAN HERNANDO URREGO ESE,COLOMBIA,DENGUE,CASANARE,AGUAZUL,COLOMBIA,CASANARE,AGUAZUL,CASANARE,AGUAZUL
2,8915477,210,2022-04-08,13,2022,8500100001,3,19,1,170,COLOMBIA ...,F,170,85,410,1,9999,S,EPS025,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,85,410,85,85001,2022-04-07,2022-04-01,2,1,2022-04-07,1,NaN,3,2002-10-09,NaN,NaN,2022-06-15,2022-06-08,NaN,NaN,NaN,1,34,1,3,Confirmado por laboratorio,HOSPITAL DE YOPAL ESE NUEVA SEDE,COLOMBIA,DENGUE,CASANARE,TAURAMENA,COLOMBIA,CASANARE,TAURAMENA,CASANARE,YOPAL
3,8908033,210,2022-04-04,13,2022,8500100001,3,8,1,170,COLOMBIA ...,M,170,85,1,1,9999,C,EPS044,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,85,1,85,85001,2022-04-03,2022-03-31,2,1,2022-04-03,1,NaN,3,2013-11-12,NaN,NaN,2022-06-01,2022-05-26,NaN,NaN,NaN,1,20769,1,3,Confirmado por laboratorio,HOSPITAL DE YOPAL ESE NUEVA SEDE,COLOMBIA,DENGUE,CASANARE,YOPAL,COLOMBIA,CASANARE,YOPAL,CASANARE,YOPAL
4,8908034,210,2022-05-16,19,2022,5000600169,1,8,1,170,COLOMBIA ...,M,170,50,6,3,9997,S,EPSS34,6,NaN,...,2,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,50,6,50,50006,2022-05-16,2022-05-12,2,1,2022-05-17,1,NaN,0,2013-11-10,NaN,NaN,2022-05-25,2022-05-18,NaN,NaN,NaN,0,20770,1,2,Probable,HOSPITAL MUNICIPAL DE ACACIAS,COLOMBIA,DENGUE,META,ACACIAS,COLOMBIA,META,ACACIAS,META,ACACIAS


In [ ]:
vistazo(df_2022)

📊 La base de datos tiene 65,691 registros y 73 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65691 entries, 0 to 65690
Data columns (total 73 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CONSECUTIVE                65691 non-null  int64  
 1   COD_EVE                    65691 non-null  int64  
 2   FEC_NOT                    65691 non-null  object 
 3   SEMANA                     65691 non-null  int64  
 4   ANO                        65691 non-null  int64  
 5   COD_PRE                    65691 non-null  int64  
 6   COD_SUB                    65691 non-null  int64  
 7   EDAD                       65691 non-null  int64  
 8   UNI_MED                    65691 non-null  int64  
 9   nacionalidad               65691 non-null  int64  
 10  nombre_nacionalidad        65691 non-null  object 
 11  SEXO                       65691 non-null  object 
 12  COD_PAIS_O    

In [ ]:
faltantes_2022 = Resumen_Faltantes(df_2022)
faltantes_2022

,faltantes,porcentaje
GRU_POB,65691,100.000
CBMTE,65691,100.000
CER_DEF,65691,100.000
FEC_DEF,65691,100.000
FM_GRADO,65334,99.457
FM_UNIDAD,65334,99.457
FM_FUERZA,65333,99.455
FEC_HOS,30989,47.174
COD_ASE,2225,3.387
FECHA_NTO,18,0.027


### **1.3.13. Datos 2023**

In [ ]:
df_2023 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2023_210.xlsx")
df_2023.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,consecutive_origen,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,10096307,210,2023-12-11,48,2023,7600108077,1,23,1,170,COLOMBIA ...,M,170,76,1,1,53291.00,C,ESSC62,6,NaN,...,1,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,76,1,76,76001,2023-12-02,2023-11-27,2,1,2023-12-02,1,NaN,3,2000-09-10,NaN,NaN,2023-12-11,2023-12-11,NaN,NaN,NaN,1,43477,1,3,Confirmado por laboratorio,CLINICA COLOMBIA,COLOMBIA,DENGUE,VALLE,CALI,COLOMBIA,VALLE,CALI,VALLE,CALI
1,10096341,210,2023-12-05,48,2023,7600109043,1,8,1,170,COLOMBIA ...,M,170,76,1,1,99999.06,C,EPS005,6,NaN,...,3,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,76,1,76,76001,2023-12-05,2023-12-02,3,1,2023-12-05,1,NaN,0,2015-02-28,NaN,NaN,2023-12-07,2023-12-07,NaN,NaN,NaN,1,43511,1,3,Confirmado por laboratorio,UNIDAD ATENCION PRIMARIA SANITAS TEQUENDAMA,COLOMBIA,DENGUE,VALLE,CALI,COLOMBIA,VALLE,CALI,VALLE,CALI
2,10096340,210,2023-12-05,48,2023,7600109043,1,13,1,170,COLOMBIA ...,F,170,76,1,1,99999.05,C,EPS005,6,NaN,...,3,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,76,1,76,76001,2023-12-03,2023-11-29,3,1,2023-12-04,1,NaN,0,2010-07-10,NaN,NaN,2023-12-05,2023-12-05,NaN,NaN,NaN,1,43510,1,3,Confirmado por laboratorio,UNIDAD ATENCION PRIMARIA SANITAS TEQUENDAMA,COLOMBIA,DENGUE,VALLE,CALI,COLOMBIA,VALLE,CALI,VALLE,CALI
3,10096329,210,2023-11-28,48,2023,7600109837,1,88,1,170,COLOMBIA ...,F,170,76,1,1,99999.04,C,EPS041,6,NaN,...,3,2,2,2,2,2,,2,2,2,2,2,2,1,2,170,76,1,76,76001,2023-11-28,2023-11-26,3,1,2023-11-28,1,NaN,0,1935-07-22,NaN,NaN,2023-11-30,2023-11-30,NaN,NaN,NaN,1,43499,1,3,Confirmado por laboratorio,CLINICA DESA SAS CALI,COLOMBIA,DENGUE,VALLE,CALI,COLOMBIA,VALLE,CALI,VALLE,CALI
4,10096304,210,2023-12-21,48,2023,7600107049,9,43,1,170,COLOMBIA ...,F,170,76,1,1,52230.00,C,EPS008,6,NaN,...,3,2,2,2,2,2,,2,2,2,2,2,2,1,1,170,76,1,76,76001,2023-12-18,2023-12-01,2,2,NaN,1,NaN,0,1980-08-27,NaN,NaN,2023-12-26,2023-12-21,NaN,NaN,NaN,0,43474,1,2,Probable,VIVA 1A IPS CALLE 7,COLOMBIA,DENGUE,VALLE,CALI,COLOMBIA,VALLE,CALI,VALLE,CALI


In [ ]:
vistazo(df_2023)

📊 La base de datos tiene 126,411 registros y 73 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126411 entries, 0 to 126410
Data columns (total 73 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CONSECUTIVE                126411 non-null  int64  
 1   COD_EVE                    126411 non-null  int64  
 2   FEC_NOT                    126411 non-null  object 
 3   SEMANA                     126411 non-null  int64  
 4   ANO                        126411 non-null  int64  
 5   COD_PRE                    126411 non-null  int64  
 6   COD_SUB                    126411 non-null  int64  
 7   EDAD                       126411 non-null  int64  
 8   UNI_MED                    126411 non-null  int64  
 9   nacionalidad               126411 non-null  int64  
 10  nombre_nacionalidad        126411 non-null  object 
 11  SEXO                       126411 non-null  object 
 1

In [ ]:
faltantes_2023 = Resumen_Faltantes(df_2023)
faltantes_2023

,faltantes,porcentaje
GRU_POB,126411,100.000
CER_DEF,126411,100.000
FEC_DEF,126411,100.000
CBMTE,126411,100.000
FM_UNIDAD,125819,99.532
FM_GRADO,125819,99.532
FM_FUERZA,125819,99.532
FEC_HOS,69793,55.211
Nom_upgd,3059,2.420
COD_ASE,2953,2.336


### **1.3.14. Datos 2024**

In [ ]:
df_2024 = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Datos_Dengue/Datos_2024_210.xlsx")
df_2024.head()

,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,nacionalidad,nombre_nacionalidad,SEXO,COD_PAIS_O,COD_DPTO_O,COD_MUN_O,AREA,OCUPACION,TIP_SS,COD_ASE,PER_ETN,GRU_POB,nom_grupo,estrato,GP_DISCAPA,GP_DESPLAZ,GP_MIGRANT,GP_CARCELA,GP_GESTAN,sem_ges,GP_INDIGEN,GP_POBICFB,GP_MAD_COM,GP_DESMOVI,GP_PSIQUIA,GP_VIC_VIO,GP_OTROS,fuente,COD_PAIS_R,COD_DPTO_R,COD_MUN_R,COD_DPTO_N,COD_MUN_N,FEC_CON,INI_SIN,TIP_CAS,PAC_HOS,FEC_HOS,CON_FIN,FEC_DEF,AJUSTE,FECHA_NTO,CER_DEF,CBMTE,FEC_ARC_XL,FEC_AJU,FM_FUERZA,FM_UNIDAD,FM_GRADO,confirmados,va_sispro,Estado_final_de_caso,nom_est_f_caso,Nom_upgd,Pais_ocurrencia,Nombre_evento,Departamento_ocurrencia,Municipio_ocurrencia,Pais_residencia,Departamento_residencia,Municipio_residencia,Departamento_Notificacion,Municipio_notificacion
0,10795018,210,2024-09-02,35,2024,7327000880,1,11,1,170,COLOMBIA,F,170,73,270,1,99999.05,C,EPS005,6,NaN,NaN,2.0,2,2,2,2,2,NaN,2,2,2,2,2,2,1,1,170,73,270,73,73270,2024-09-02,2024-08-30,2,2,NaN,1,NaN,0,2013-01-09,NaN,NaN,2024-09-03,2024-09-03,NaN,NaN,NaN,0,1,2,Probable,HOSPITAL SANTA ANA,COLOMBIA,DENGUE,TOLIMA,FALAN,COLOMBIA,TOLIMA,FALAN,TOLIMA,FALAN
1,10795636,210,2024-09-13,37,2024,6800170276,2,8,1,170,COLOMBIA,M,170,68,1,1,99999.05,S,EPSS05,6,NaN,NaN,1.0,2,2,2,2,2,NaN,2,2,2,2,2,2,1,1,170,68,1,68,68001,2024-09-11,2024-09-08,3,2,NaN,1,NaN,0,2016-09-09,NaN,NaN,2024-09-14,2024-09-14,NaN,NaN,NaN,1,1,3,Confirmado por laboratorio,UNIDAD DE URGENCIAS BUCARAMANGA BOLARQUI,COLOMBIA,DENGUE,SANTANDER,BUCARAMANGA,COLOMBIA,SANTANDER,BUCARAMANGA,SANTANDER,BUCARAMANGA
2,10795673,210,2024-08-08,32,2024,6800170276,2,7,1,170,COLOMBIA,M,170,68,1,1,99999.05,C,EPS005,6,NaN,NaN,2.0,2,2,2,2,2,NaN,2,2,2,2,2,2,1,1,170,68,1,68,68001,2024-08-08,2024-08-05,3,2,NaN,1,NaN,0,2017-06-19,NaN,NaN,2024-08-11,2024-08-11,NaN,NaN,NaN,1,1,3,Confirmado por laboratorio,UNIDAD DE URGENCIAS BUCARAMANGA BOLARQUI,COLOMBIA,DENGUE,SANTANDER,BUCARAMANGA,COLOMBIA,SANTANDER,BUCARAMANGA,SANTANDER,BUCARAMANGA
3,10794970,210,2024-03-26,12,2024,6800104268,1,20,1,170,COLOMBIA,F,170,68,1,1,51424.00,C,EPS002,6,NaN,NaN,2.0,2,2,2,2,2,NaN,2,2,2,2,2,2,1,1,170,68,1,68,68001,2024-03-23,2024-03-22,2,1,2024-03-30,1,NaN,3,2004-03-14,NaN,NaN,2024-03-26,2024-03-26,NaN,NaN,NaN,1,1,3,Confirmado por laboratorio,CLINICA DE URGENCIAS BUCARAMANGA SAS,COLOMBIA,DENGUE,SANTANDER,BUCARAMANGA,COLOMBIA,SANTANDER,BUCARAMANGA,SANTANDER,BUCARAMANGA
4,10794977,210,2024-05-25,20,2024,6800103671,1,19,1,170,COLOMBIA,M,170,68,1,1,99999.05,P,RES004,6,NaN,NaN,3.0,2,2,2,2,2,NaN,2,2,2,2,2,2,1,1,170,68,1,68,68001,2024-05-23,2024-05-14,2,2,NaN,1,NaN,3,2004-06-24,NaN,NaN,2024-07-04,2024-07-04,NaN,NaN,NaN,1,1,3,Confirmado por laboratorio,FUNDACION AVANZAR FOS,COLOMBIA,DENGUE,SANTANDER,BUCARAMANGA,COLOMBIA,SANTANDER,BUCARAMANGA,SANTANDER,BUCARAMANGA


In [ ]:
vistazo(df_2024)

📊 La base de datos tiene 309,627 registros y 72 variables

🔎 Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309627 entries, 0 to 309626
Data columns (total 72 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CONSECUTIVE                309627 non-null  int64  
 1   COD_EVE                    309627 non-null  int64  
 2   FEC_NOT                    309627 non-null  object 
 3   SEMANA                     309627 non-null  int64  
 4   ANO                        309627 non-null  int64  
 5   COD_PRE                    309627 non-null  int64  
 6   COD_SUB                    309627 non-null  int64  
 7   EDAD                       309627 non-null  int64  
 8   UNI_MED                    309627 non-null  int64  
 9   nacionalidad               309627 non-null  int64  
 10  nombre_nacionalidad        309627 non-null  object 
 11  SEXO                       309627 non-null  object 
 1

In [ ]:
faltantes_2024 = Resumen_Faltantes(df_2024)
faltantes_2024

,faltantes,porcentaje
GRU_POB,309627,100.000
CER_DEF,309627,100.000
CBMTE,309627,100.000
FEC_DEF,309627,100.000
sem_ges,308003,99.475
FM_GRADO,307588,99.341
FM_UNIDAD,307588,99.341
FM_FUERZA,307586,99.341
nom_grupo,306292,98.923
FEC_HOS,203001,65.563


## **1.4. Análisis de Faltantes:**

En esta subsección se analizará la presencia de valores faltantes en los datos cargados, con el objetivo de identificar aquellas variables que presentan porcentajes elevados de ausencia de información. Dichas variables serán evaluadas para su posible exclusión del análisis, considerando que niveles excesivos de datos faltantes pueden comprometer la calidad, consistencia y viabilidad del tratamiento estadístico y modelado posterior.

In [50]:
faltantes_2011

,faltantes,porcentaje
nacionalidad,29386,100.000
nombre_nacionalidad,29386,100.000
nom_grupo,29386,100.000
estrato,29386,100.000
sem_ges,29386,100.000
fuente,29386,100.000
COD_PAIS_R,29386,100.000
FM_UNIDAD,29386,100.000
FM_GRADO,29386,100.000
Pais_residencia,29386,100.000


In [51]:
resumenes = {
    "2011": faltantes_2011, 
    "2012": faltantes_2012,
    "2013": faltantes_2013,
    "2014": faltantes_2014,
    "2015": faltantes_2015,
    "2016": faltantes_2016,
    "2017": faltantes_2017,
    "2018": faltantes_2018,
    "2019": faltantes_2019,
    "2020": faltantes_2020,
    "2021": faltantes_2021,
    "2022": faltantes_2022,
    "2023": faltantes_2023,
    "2024": faltantes_2024
}


# Paso 1: obtener los nombres de variables con faltantes en cada año
sets_faltantes = {anio: set(df.index) for anio, df in resumenes.items()}

# Paso 2: intersección de todas las variables con faltantes
comunes = set.intersection(*sets_faltantes.values())

print("Variables con faltantes en TODOS los años:")
print(comunes)

Variables con faltantes en TODOS los años:
{'CBMTE', 'FEC_HOS', 'COD_ASE', 'CER_DEF', 'FM_GRADO', 'FM_UNIDAD', 'FEC_DEF'}


In [52]:
df_2011 = eliminar_por_faltantes(df_2011,faltantes_2011)
Resumen_Faltantes(df_2011)

Eliminando 14 variables


,faltantes,porcentaje
FEC_HOS,18544,63.105
COD_ASE,3259,11.090
FECHA_NTO,959,3.263
INI_SIN,3,0.010


In [53]:
df_2012 = eliminar_por_faltantes(df_2012, faltantes_2012)
Resumen_Faltantes(df_2012)

Eliminando 14 variables


,faltantes,porcentaje
FEC_HOS,35479,67.622
COD_ASE,3783,7.210
FECHA_NTO,582,1.109
INI_SIN,15,0.029


In [54]:
df_2013 = eliminar_por_faltantes(df_2013, faltantes_2013)
Resumen_Faltantes(df_2013)

Eliminando 14 variables


,faltantes,porcentaje
FEC_HOS,75427,61.603
COD_ASE,7284,5.949
FECHA_NTO,1211,0.989
INI_SIN,28,0.023
FEC_CON,11,0.009


In [55]:
df_2014 = eliminar_por_faltantes(df_2014, faltantes_2014)
Resumen_Faltantes(df_2014)

Eliminando 15 variables


,faltantes,porcentaje
FEC_HOS,68066,64.606
COD_ASE,5093,4.834
FECHA_NTO,740,0.702
FEC_CON,25,0.024


In [56]:
df_2015 = eliminar_por_faltantes(df_2015, faltantes_2015)
Resumen_Faltantes(df_2015)

Eliminando 15 variables


,faltantes,porcentaje
FEC_HOS,61966,65.212
COD_ASE,3237,3.407
FECHA_NTO,269,0.283
FEC_CON,28,0.029


In [57]:
df_2016 = eliminar_por_faltantes(df_2016, faltantes_2016)
Resumen_Faltantes(df_2016)

Eliminando 16 variables


,faltantes,porcentaje
COD_ASE,3901,3.896
FECHA_NTO,289,0.289
FEC_CON,24,0.024
INI_SIN,24,0.024


In [58]:
df_2017 = eliminar_por_faltantes(df_2017, faltantes_2017)
Resumen_Faltantes(df_2017)

Eliminando 16 variables


,faltantes,porcentaje
FEC_HOS,16230,64.796
COD_ASE,445,1.777
FEC_CON,2,0.008


In [59]:
df_2018 = eliminar_por_faltantes(df_2018, faltantes_2018)
Resumen_Faltantes(df_2018)

Eliminando 12 variables


,faltantes,porcentaje
FEC_HOS,22333,51.161
COD_ASE,1565,3.585
FECHA_NTO,109,0.250


In [60]:
df_2019 = eliminar_por_faltantes(df_2019, faltantes_2019)
Resumen_Faltantes(df_2019)

Eliminando 8 variables


,faltantes,porcentaje
FEC_HOS,63805,51.605
COD_ASE,3948,3.193
FECHA_NTO,158,0.128
FM_FUERZA,1,0.001


In [61]:
df_2020 = eliminar_por_faltantes(df_2020, faltantes_2020)
Resumen_Faltantes(df_2020)

Eliminando 6 variables


,faltantes,porcentaje
FEC_HOS,41924,54.861
COD_ASE,3729,4.880
FECHA_NTO,84,0.110


In [62]:
df_2021 = eliminar_por_faltantes(df_2021, faltantes_2021)
Resumen_Faltantes(df_2021)

Eliminando 7 variables


,faltantes,porcentaje
FEC_HOS,24655,49.985
COD_ASE,2049,4.154
Municipio_residencia,4,0.008


In [63]:
df_2022 = eliminar_por_faltantes(df_2022, faltantes_2022)
Resumen_Faltantes(df_2022)

Eliminando 7 variables


,faltantes,porcentaje
FEC_HOS,30989,47.174
COD_ASE,2225,3.387
FECHA_NTO,18,0.027
Municipio_residencia,2,0.003
FEC_CON,1,0.002


In [64]:
df_2023 = eliminar_por_faltantes(df_2023, faltantes_2023)
Resumen_Faltantes(df_2023)

Eliminando 7 variables


,faltantes,porcentaje
FEC_HOS,69793,55.211
Nom_upgd,3059,2.420
COD_ASE,2953,2.336
FECHA_NTO,199,0.157


In [65]:
df_2024 = eliminar_por_faltantes(df_2024, faltantes_2024)
Resumen_Faltantes(df_2024)

Eliminando 9 variables


,faltantes,porcentaje
FEC_HOS,203001,65.563
estrato,13144,4.245
COD_ASE,4639,1.498
FECHA_NTO,89,0.029


## **1.5. Concatenar Datasets**

En esta etapa se procederá a la concatenación de los distintos conjuntos de datos, con el fin de construir una única base consolidada que integre la información correspondiente al periodo 2011–2024. Para esto, se realizará la normalización de los nombres de las columnas para garantizar la consistencia estructural entre los diferentes archivos. Asimismo, se seleccionarán las variables relevantes para el desarrollo del análisis y se generará la base de datos preliminar que servirá como insumo para las etapas posteriores del estudio.

In [66]:
dfs_name = dfs = ['df_2011','df_2012', 'df_2013', 'df_2014', 'df_2015', 'df_2016', 'df_2017', 'df_2018','df_2019', 'df_2020', 'df_2021', 'df_2022', 'df_2023', 'df_2024']
dfs = [df_2011,df_2012, df_2013, df_2014, df_2015, df_2016, df_2017, df_2018 ,df_2019, df_2020, df_2021, df_2022, df_2023, df_2024]

In [ ]:
for df in dfs:
    df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

/tmp/ipykernel_275816/3294264302.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r'^\s*$', np.nan, regex=True)
/tmp/ipykernel_275816/3294264302.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r'^\s*$', np.nan, regex=True)
/tmp/ipykernel_275816/3294264302.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_op

In [71]:
df_2011 = procesar_df(df_2011)
df_2012 = procesar_df(df_2012)
df_2013 = procesar_df(df_2013)
df_2014 = procesar_df(df_2014)
df_2015 = procesar_df(df_2015)
df_2016 = procesar_df(df_2016)
df_2017 = procesar_df(df_2017)
df_2018 = procesar_df(df_2018)
df_2019 = procesar_df(df_2019)
df_2020 = procesar_df(df_2020)
df_2021 = procesar_df(df_2021)
df_2022 = procesar_df(df_2022)
df_2023 = procesar_df(df_2023)
df_2024 = procesar_df(df_2024)

/tmp/ipykernel_275816/3294264302.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r'^\s*$', np.nan, regex=True)
/tmp/ipykernel_275816/3294264302.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r'^\s*$', np.nan, regex=True)
/tmp/ipykernel_275816/3294264302.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_op

In [73]:
dfs = [df_2011,df_2012, df_2013, df_2014, df_2015, df_2016, df_2017, df_2018 ,df_2019, df_2020, df_2021, df_2022, df_2023, df_2024]

In [74]:
for df, name in zip(dfs, dfs_name):
    print(f"\n{name}")
    print(df.columns)
    print('='*80)


df_2011
Index(['ajuste', 'ano', 'area', 'cod_ase', 'cod_dpto_n', 'cod_dpto_o',
       'cod_dpto_r', 'cod_eve', 'cod_mun_n', 'cod_mun_o', 'cod_mun_r',
       'cod_pais_o', 'cod_pre', 'cod_sub', 'con_fin', 'confirmado',
       'consecutive', 'consecutive_origen', 'departamento_notificacion',
       'departamento_ocurrencia', 'departamento_residencia', 'edad',
       'estado_final_de_caso', 'fec_aju', 'fec_arc_xl', 'fec_con', 'fec_hos',
       'fec_not', 'fecha_nto', 'gp_carcela', 'gp_desmovi', 'gp_desplaz',
       'gp_discapa', 'gp_gestan', 'gp_indigen', 'gp_mad_com', 'gp_migrant',
       'gp_otros', 'gp_pobicfb', 'gp_psiquia', 'gp_vic_vio', 'gru_pob',
       'ini_sin', 'municipio_notificacion', 'municipio_ocurrencia',
       'municipio_residencia', 'nom_est_f_caso', 'nombre_evento',
       'nombre_upgd', 'ocupacion', 'pac_hos', 'pais_ocurrencia', 'particin',
       'per_etn', 'semana', 'sexo', 'tip_cas', 'tip_ss', 'uni_med',
       'va_sispro'],
      dtype='object')

df_2012
Index(['a

In [75]:
for df, name in zip(dfs,dfs_name):
    print('\n',name)
    print(Resumen_Faltantes(df))
    print('='*80)


 df_2011
           faltantes  porcentaje
fec_hos        18544      63.105
cod_ase         3259      11.090
fecha_nto        959       3.263
ajuste           604       2.055
ini_sin            3       0.010

 df_2012
           faltantes  porcentaje
fec_hos        35479      67.622
cod_ase         3783       7.210
ajuste          1126       2.146
fecha_nto        582       1.109
ini_sin           15       0.029

 df_2013
           faltantes  porcentaje
fm_fuerza     121184      98.973
fec_hos        75427      61.603
cod_ase         7284       5.949
ajuste          3785       3.091
fecha_nto       1211       0.989
ini_sin           28       0.023
fec_con           11       0.009

 df_2014
           faltantes  porcentaje
fec_hos        68066      64.606
cod_ase         5093       4.834
ajuste          4065       3.858
fecha_nto        740       0.702
fec_con           25       0.024

 df_2015
           faltantes  porcentaje
fec_hos        61966      65.212
ajuste          3256      

Con la anterior revision realizada tomamos la decision de seleccionar unas variables particulares que consideramos relevantes para nuestro análisis.

In [76]:
columnas = [
    'ano', 'area', 'edad', 'sexo', 'semana',
    'departamento_notificacion', 'departamento_ocurrencia', 'municipio_ocurrencia',
    'tip_cas', 'confirmado', 'pac_hos', 'fec_not', 'ini_sin', 'fec_con',
    'per_etn', 'tip_ss', 'estado_final_de_caso', 'nombre_evento',
    'uni_med', 'cod_pais_o', 'consecutive', 'pais_ocurrencia','cod_dpto_o'
]

dfs = [df_2011,df_2012, df_2013, df_2014, df_2015, df_2016, df_2017,
       df_2018, df_2019, df_2020, df_2021, df_2022, df_2023, df_2024]

df_dengue = pd.concat(
    [df[columnas] for df in dfs],
    ignore_index=True
)

In [78]:
df_dengue.head()

,ano,area,edad,sexo,semana,departamento_notificacion,departamento_ocurrencia,municipio_ocurrencia,tip_cas,confirmado,pac_hos,fec_not,ini_sin,fec_con,per_etn,tip_ss,estado_final_de_caso,nombre_evento,uni_med,cod_pais_o,consecutive,pais_ocurrencia,cod_dpto_o
0,2011,Cabecera,4,F,6,CESAR,CESAR,AGUSTIN CODAZZI,Probable,No confirmado,Sí,2011-02-11,2011-02-09,2011-02-11,6,Subsidiado,2,DENGUE,Años,170,4265187,COLOMBIA,20
1,2011,Cabecera,31,F,41,VALLE,VALLE,TULUA,Confirmado Lab,Confirmado,Sí,2011-10-15,2011-10-12,2011-10-15,6,Contributivo,3,DENGUE,Años,170,4265188,COLOMBIA,76
2,2011,Cabecera,47,M,27,CORDOBA,CORDOBA,MONTERIA,Confirmado Lab,Confirmado,Sí,2011-07-11,2011-07-03,2011-07-11,6,Subsidiado,3,DENGUE,Años,170,4265189,COLOMBIA,23
3,2011,Rural,5,M,38,TOLIMA,TOLIMA,PURIFICACION,Probable,No confirmado,No,2011-09-25,2011-09-24,2011-09-25,6,Subsidiado,2,DENGUE,Meses,170,4260779,COLOMBIA,73
4,2011,Cabecera,39,M,1,CHOCO,CHOCO,SIPI,Confirmado Lab,Confirmado,No,2011-01-15,2011-01-08,2011-01-15,5,No asegurado,3,DENGUE,Años,170,4270131,COLOMBIA,27


In [80]:
df_dengue.to_csv("/home/guirlessa/Dl_Proyecto_Dengue/Bases_Datos/df_2012_2024.csv", index=False)

## **1.6. Revisando Nueva Base de Datos**

Tras la integración de los distintos conjuntos de datos correspondientes al periodo de estudio y la selección de las variables pertinentes para la consolidación de la base analítica, el siguiente paso consiste en evaluar rigurosamente la calidad de la información recopilada. Posteriormente, se procederá a la depuración y estandarización de la base de datos, con el propósito de construir el Dataset final que servirá como fundamento para los análisis posteriores.

In [8]:
df = pd.read_csv("/home/guirlessa/Dl_Proyecto_Dengue/Bases_Datos/df_2012_2024.csv")
df.head()

,ano,area,edad,sexo,semana,departamento_notificacion,departamento_ocurrencia,municipio_ocurrencia,tip_cas,confirmado,...,fec_con,per_etn,tip_ss,estado_final_de_caso,nombre_evento,uni_med,cod_pais_o,consecutive,pais_ocurrencia,cod_dpto_o
0,2011,Cabecera,4,F,6,CESAR,CESAR,AGUSTIN CODAZZI,Probable,No confirmado,...,2011-02-11,6,Subsidiado,2,DENGUE,Años,170,4265187,COLOMBIA,20
1,2011,Cabecera,31,F,41,VALLE,VALLE,TULUA,Confirmado Lab,Confirmado,...,2011-10-15,6,Contributivo,3,DENGUE,Años,170,4265188,COLOMBIA,76
2,2011,Cabecera,47,M,27,CORDOBA,CORDOBA,MONTERIA,Confirmado Lab,Confirmado,...,2011-07-11,6,Subsidiado,3,DENGUE,Años,170,4265189,COLOMBIA,23
3,2011,Rural,5,M,38,TOLIMA,TOLIMA,PURIFICACION,Probable,No confirmado,...,2011-09-25,6,Subsidiado,2,DENGUE,Meses,170,4260779,COLOMBIA,73
4,2011,Cabecera,39,M,1,CHOCO,CHOCO,SIPI,Confirmado Lab,Confirmado,...,2011-01-15,5,No asegurado,3,DENGUE,Años,170,4270131,COLOMBIA,27


In [9]:
print(f'El nuevo dataset tiene {df.shape[0]} registros y {df.shape[1]} variables')

El nuevo dataset tiene 1324604 registros y 23 variables


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324604 entries, 0 to 1324603
Data columns (total 23 columns):
 #   Column                     Non-Null Count    Dtype 
---  ------                     --------------    ----- 
 0   ano                        1324604 non-null  int64 
 1   area                       1324604 non-null  object
 2   edad                       1324604 non-null  int64 
 3   sexo                       1324604 non-null  object
 4   semana                     1324604 non-null  int64 
 5   departamento_notificacion  1324604 non-null  object
 6   departamento_ocurrencia    1324604 non-null  object
 7   municipio_ocurrencia       1324604 non-null  object
 8   tip_cas                    1324604 non-null  object
 9   confirmado                 1324604 non-null  object
 10  pac_hos                    1324604 non-null  object
 11  fec_not                    1324604 non-null  object
 12  ini_sin                    1324534 non-null  object
 13  fec_con                    

A partir de esta inspección inicial mediante `df.info()`, se evidencia la necesidad de corregir y estandarizar los tipos de datos de algunas variables, con el fin de garantizar un procesamiento adecuado y continuar de manera consistente con las etapas posteriores del análisis.


In [11]:
for col in ['fec_not', 'fec_not', 'fec_con']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['semana'] = df['semana'].astype('int8')

categoricas = [
    'area', 'sexo', 'departamento_notificacion', 'departamento_ocurrencia',
    'municipio_ocurrencia', 'tip_cas', 'confirmado', 'pac_hos',
    'tip_ss', 'nombre_evento', 'uni_med', 'pais_ocurrencia',
    'per_etn', 'estado_final_de_caso', 'cod_pais_o','cod_dpto_o'
]

for col in categoricas:
    df[col] = df[col].astype('category')

print(df.dtypes)

ano                                   int64
area                               category
edad                                  int64
sexo                               category
semana                                 int8
departamento_notificacion          category
departamento_ocurrencia            category
municipio_ocurrencia               category
tip_cas                            category
confirmado                         category
pac_hos                            category
fec_not                      datetime64[ns]
ini_sin                              object
fec_con                      datetime64[ns]
per_etn                            category
tip_ss                             category
estado_final_de_caso               category
nombre_evento                      category
uni_med                            category
cod_pais_o                         category
consecutive                           int64
pais_ocurrencia                    category
cod_dpto_o                      

A continuación, se procederá a verificar la existencia de registros duplicados y valores faltantes en la base de datos, con el propósito de evaluar la calidad e integridad de la información antes de continuar con el análisis.

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
Resumen_Faltantes(df)

,faltantes,porcentaje
fec_con,91,0.007
ini_sin,70,0.005


Los resultados evidencian una proporción mínima de valores faltantes en las variables `fec_con` e `ini_sin`, con porcentajes inferiores al 0.01 %, lo que indica una alta completitud y calidad de la información registrada.

### **1.6.2. Limpieza del Dataset**

#### **1.6.2.1. Filtrado del Dataset**

In [14]:
df['pais_ocurrencia'].unique()

['COLOMBIA', 'VENEZUELA', 'BRASIL', 'ECUADOR', 'ESTADOS UNIDOS DE AMERICA', ..., 'BARBADOS', 'SAN PEDRO Y MIGUELON', 'GAMBIA', 'ALBANIA', 'EL SALVADOR']
Length: 67
Categories (67, object): ['AFGANISTAN', 'ALBANIA', 'ALEMANIA', 'ARGENTINA', ..., 'TAILANDIA', 'TOGO', 'URUGUAY', 'VENEZUELA']

In [15]:
df['pais_ocurrencia'].value_counts()

pais_ocurrencia
COLOMBIA                1321714
VENEZUELA                  2087
BRASIL                      298
PERU                         89
MEXICO                       69
                         ...   
NICARAGUA                     1
SUIZA                         1
SAN PEDRO Y MIGUELON          1
TOGO                          1
TAILANDIA                     1
Name: count, Length: 67, dtype: int64

Se identificó la presencia de registros correspondientes a casos de dengue ocurridos en países distintos de Colombia. No obstante, dado que el presente estudio se centra en el análisis del comportamiento de la incidencia del dengue en Colombia, se realizará un filtrado de la base de datos para conservar únicamente los registros asociados al territorio nacional.

In [16]:
df = df[df['pais_ocurrencia'] == 'COLOMBIA'].copy()
df['pais_ocurrencia'] = df['pais_ocurrencia'].cat.remove_unused_categories()
df['pais_ocurrencia'].value_counts()

pais_ocurrencia
COLOMBIA    1321714
Name: count, dtype: int64

De igual manera, dado que el objetivo del estudio es analizar el comportamiento temporal de la incidencia del dengue, se conservarán únicamente los casos positivos, es decir, aquellos registros clasificados como confirmados dentro del sistema de vigilancia epidemiológica.

In [17]:
df = df[df['confirmado'] == 'Confirmado'].copy()
df['confirmado'] = df['confirmado'].cat.remove_unused_categories()
df['confirmado'].value_counts()

confirmado
Confirmado    806788
Name: count, dtype: int64

Posteriormente, se realizó un proceso de filtrado sobre la variable `uni_med`, conservando únicamente los registros expresados en años y meses, debido a su relevancia para el análisis. Adicionalmente, se eliminaron categorías no utilizadas en las variables categóricas con el fin de optimizar la consistencia y limpieza de la base de datos.


In [18]:
df = df[df['uni_med'].isin(['Años', 'Meses'])].copy()

for col in df.select_dtypes(include='category').columns:
    df[col] = df[col].cat.remove_unused_categories()

df['uni_med'].value_counts()

uni_med
Años     791066
Meses     15224
Name: count, dtype: int64

#### **1.6.2.2. Limpiando Columna Departamento**

In [19]:
df['departamento_ocurrencia'].unique()

['VALLE', 'CORDOBA', 'CHOCO', 'GUAJIRA', 'ATLANTICO', ..., 'AMAZONAS', 'QUINDIO', 'GUAINIA', 'VAUPES', 'BOGOTA']
Length: 35
Categories (35, object): ['AMAZONAS', 'ANTIOQUIA', 'ARAUCA', 'ATLANTICO', ..., 'TOLIMA', 'VALLE', 'VAUPES', 'VICHADA']

Asimismo, se procederá a homogenizar la columna correspondiente a los departamentos, con el fin de garantizar la consistencia de los registros y facilitar su integración con otras bases de datos en etapas posteriores del análisis.

In [20]:
df['departamento'] = df['departamento_notificacion'].str.upper()
df['departamento'].unique()

df = df[~df['departamento'].isin(['EXTERIOR', 'PROCEDENCIA DESCONOCIDA'])].copy()

mapeo_dptos = {
    'AMAZONAS'               : 'Amazonas',
    'ANTIOQUIA'              : 'Antioquia',
    'ARAUCA'                 : 'Arauca',
    'ATLANTICO'              : 'Atlantico',
    'BOGOTA'                 : 'Bogota d.c.',
    'BOLIVAR'                : 'Bolivar',
    'BOYACA'                 : 'Boyaca',
    'CALDAS'                 : 'Caldas',
    'CAQUETA'                : 'Caqueta',
    'CASANARE'               : 'Casanare',
    'CAUCA'                  : 'Cauca',
    'CESAR'                  : 'Cesar',
    'CHOCO'                  : 'Choco',
    'CORDOBA'                : 'Cordoba',
    'CUNDINAMARCA'           : 'Cundinamarca',
    'GUAINIA'                : 'Guainia',
    'GUAJIRA'                : 'Guajira',
    'GUAVIARE'               : 'Guaviare',
    'HUILA'                  : 'Huila',
    'MAGDALENA'              : 'Magdalena',
    'META'                   : 'Meta',
    'NARIÑO'                 : 'Narino',
    'NORTE SANTANDER'        : 'Norte De Santander',
    'PUTUMAYO'               : 'Putumayo',
    'QUINDIO'                : 'Quindio',
    'RISARALDA'              : 'Risaralda',
    'SAN ANDRES'             : 'San Andres Y Providencia',
    'SANTANDER'              : 'Santander',
    'SUCRE'                  : 'Sucre',
    'TOLIMA'                 : 'Tolima',
    'VALLE'                  : 'Valle Del Cauca',
    'VAUPES'                 : 'Vaupes',
    'VICHADA'                : 'Vichada'
}

df['departamento'] = (
    df['departamento']
    .map(mapeo_dptos)
    .astype('category')
)

df['departamento'] = df['departamento'].cat.remove_unused_categories()

df['departamento'].unique()

['Valle Del Cauca', 'Cordoba', 'Choco', 'Casanare', 'Atlantico', ..., 'Cauca', 'Amazonas', 'Quindio', 'Guainia', 'Vaupes']
Length: 33
Categories (33, object): ['Amazonas', 'Antioquia', 'Arauca', 'Atlantico', ..., 'Tolima', 'Valle Del Cauca', 'Vaupes', 'Vichada']

In [21]:
df.shape

(806290, 24)

## **1.7. Generando Base de Datos Final**

Posteriormente, se genera un nuevo conjunto de datos con el objetivo de agrupar los registros por semana epidemiológica, en concordancia con la metodología planteada en el proyecto. Adicionalmente, se realizan las respectivas verificaciones sobre las variables correspondientes a departamentos y años, con el fin de garantizar la consistencia y calidad de la información antes de continuar con el análisis.

In [22]:
df_grouped = (
    df.groupby(['departamento', 'ano', 'semana'])
      .size()
      .reset_index(name='casos')
)

/tmp/ipykernel_41799/2105460455.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['departamento', 'ano', 'semana'])


In [23]:
df_grouped.shape

(24486, 4)

In [24]:
df_grouped.head()

,departamento,ano,semana,casos
0,Amazonas,2011,1,0
1,Amazonas,2011,2,0
2,Amazonas,2011,3,1
3,Amazonas,2011,4,3
4,Amazonas,2011,5,0


In [25]:
anos = df_grouped['ano'].unique()
anos

array([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
       2022, 2023, 2024])

### **1.7.1. Estructuración de la base de datos por semana epidemiológica**

En esta etapa se genera una secuencia continua de fechas comprendida entre los años 2011 y 2024. Posteriormente, a cada fecha se le asigna el año y la semana epidemiológica correspondiente utilizando el calendario ISO, el cual considera el lunes como el inicio de la semana. Este procedimiento permite estructurar temporalmente los datos y facilitar su posterior agregación y análisis por semanas epidemiológicas.


In [26]:
fechas = pd.date_range(
    start="2011-01-01",
    end="2024-12-31",
    freq="D"
)

tmp = pd.DataFrame({"fecha": fechas})

iso = tmp["fecha"].dt.isocalendar()

tmp["iso_year"] = iso.year
tmp["iso_week"] = iso.week

In [27]:
df1 = df
df = tmp
df["fecha"] = pd.to_datetime(df["fecha"])

iso = df["fecha"].dt.isocalendar()

df["iso_year"] = iso.year
df["iso_week"] = iso.week

Posteriormente, se construye un nuevo conjunto de datos que contiene el año, la semana epidemiológica y la fecha de inicio de cada semana. Este procedimiento permite generar una estructura temporal de referencia basada en semanas epidemiológicas únicas.

Adicionalmente, se crea una combinación completa entre todos los departamentos y cada semana del periodo de estudio mediante un índice multi-dimensional (`MultiIndex`). La utilidad de este proceso radica en garantizar que cada departamento disponga de registros para todas las semanas epidemiológicas, incluso en aquellos casos donde no se hayan reportado eventos. Esto facilita la construcción de series temporales homogéneas y continuas, condición fundamental para los análisis estadísticos y modelos de predicción posteriores.


In [28]:
df_semanal = (
    df.groupby(["iso_year", "iso_week"], as_index=False)
      .agg({
          "fecha": "min"
           })
)

In [29]:
df_semanal["fecha"] = pd.to_datetime(df_semanal["fecha"])

iso = df_semanal["fecha"].dt.isocalendar()

calendar_df = pd.DataFrame({
    "fecha_inicio": df_semanal["fecha"],
    "ano": iso.year,
    "semana": iso.week
}).drop_duplicates()

In [30]:
departamentos = df_grouped["departamento"].unique()

index = pd.MultiIndex.from_product(
    [departamentos, calendar_df.index],
    names=["departamento", "index"]
)

Seguidamente, se construye un nuevo dataframe que integra cada departamento con todas las semanas epidemiológicas del periodo de estudio. Para ello, se combinan las estructuras previamente generadas mediante un proceso de unión (`merge`), obteniendo así una base temporal completa que servirá posteriormente para incorporar el número de casos reportados por semana epidemiológica en cada departamento.


In [31]:
df_full = (
    pd.DataFrame(index=index)
    .reset_index()
    .merge(calendar_df.reset_index(), on="index")
    .drop(columns="index")
)

In [32]:
df_full = df_full.merge(
    df_grouped,
    how="left"
)

In [33]:
df_full['casos'] = df_full['casos'].fillna(0)

In [34]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos
0,Amazonas,2011-01-01,2010,52,0.0
1,Amazonas,2011-01-03,2011,1,0.0
2,Amazonas,2011-01-10,2011,2,0.0
3,Amazonas,2011-01-17,2011,3,1.0
4,Amazonas,2011-01-24,2011,4,3.0


### **1.7.2. Feature Engineering**

In [35]:
df = df1

#### **1.7.2.1. Número de Casos por Sexo**

In [36]:
df['sexo'].unique()

['F', 'M']
Categories (2, object): ['F', 'M']

In [37]:
sexo_counts = (
    df.groupby(['ano', 'semana','departamento', 'sexo'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
sexo_counts = sexo_counts.sort_values(['departamento','ano','semana'],ascending=True)
sexo_counts

sexo,ano,semana,departamento,F,M
54,2011,3,Amazonas,0,1
84,2011,4,Amazonas,1,2
169,2011,7,Amazonas,0,1
199,2011,8,Amazonas,1,4
228,2011,9,Amazonas,0,3
...,...,...,...,...,...
20675,2024,48,Vichada,8,1
20708,2024,49,Vichada,3,2
20740,2024,50,Vichada,9,4
20771,2024,51,Vichada,5,4


Ahora, se verifica la correspondencia de los nombres de los departamentos entre ambos conjuntos de datos, con el propósito de garantizar la compatibilidad necesaria para realizar el proceso de unión (`merge`). Esto permite incorporar correctamente la variable correspondiente al número de casos por sexo dentro del dataset principal.


In [38]:
set(df_full['departamento'].unique()) - set(sexo_counts['departamento'].unique())

set()

In [39]:
set(sexo_counts['semana'].unique()) - set(df_full['semana'].unique())

set()

Se observa que las columnas correspondientes a los departamentos presentan una correcta alineación entre ambos dataframes, por lo que es posible proceder con el proceso de unión (`merge`) para integrar la información de manera consistente.

In [40]:
sexo_counts = sexo_counts.rename(columns={'M': 'casos_hombres', 'F': 'casos_mujeres'})

df_full = df_full.merge(
    sexo_counts, 
    on=['ano', 'semana', 'departamento'], 
    how='left'
)


df_full[['casos_hombres', 'casos_mujeres']] = df_full[['casos_hombres', 'casos_mujeres']].fillna(0)

df_full[['casos_hombres', 'casos_mujeres']] = df_full[['casos_hombres', 'casos_mujeres']].astype(int)

df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres
0,Amazonas,2011-01-01,2010,52,0.0,0,0
1,Amazonas,2011-01-03,2011,1,0.0,0,0
2,Amazonas,2011-01-10,2011,2,0.0,0,0
3,Amazonas,2011-01-17,2011,3,1.0,0,1
4,Amazonas,2011-01-24,2011,4,3.0,1,2


#### **1.7.2.2. Número de Casos por Área**

In [41]:
df['area'].value_counts()

area
Cabecera          665522
Rural              74612
Centro Poblado     66156
Name: count, dtype: int64

La variable correspondiente al área de ocurrencia sera agrupada en dos categorías: urbano y rural. La categoría “Cabecera” se consideró urbana, mientras que “Centro Poblado” y “Rural” se agruparon como rural debido a sus características territoriales similares. Esta transformación permitira simplificar la variable, reducir la cantidad de categorías y facilitar el análisis e interpretación de los resultados.

In [42]:
df['area'] = df['area'].map({
    'Cabecera': 'Urbano',
    'Rural': 'Rural',
    'Centro Poblado': 'Rural'
}).astype('category')

In [43]:
df['area'].value_counts()

area
Urbano    665522
Rural     140768
Name: count, dtype: int64

In [44]:
area_counts = (
    df.groupby(['ano', 'semana', 'departamento','area'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
area_counts = area_counts.rename(columns={'Rural': 'casos_rural','Urbano': 'casos_urbano'})

In [45]:

df_full = df_full.merge(
    area_counts,
    on=['ano', 'semana', 'departamento'],
    how='left'
)
cols_conteo = ['casos_rural','casos_urbano']
df_full[cols_conteo] = df_full[cols_conteo].fillna(0).astype(int)

In [46]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3


#### **1.7.2.3. Número Casos por Edad**

In [47]:
df['edad_anos'] = df.apply(
    lambda row: row['edad'] / 12 if row['uni_med'] == 'Meses' else row['edad'],
    axis=1
)

bins   = [0, 5, 13, 26, 60, float('inf')]
labels = ['(0_5]', '(5_13]', '(13_26]', '(26_60]', '60_mas']

df['grupo_etario'] = pd.cut(
    df['edad_anos'],
    bins=bins,
    labels=labels,
    right=True 
)

# Paso 3: Agregar por grupo etario (Queda igual)
edad_counts = (
    df.groupby(['ano', 'semana', 'departamento', 'grupo_etario'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

df_full = df_full.merge(
    edad_counts,
    on=['ano', 'semana', 'departamento'],
    how='left'
)

cols_edad = [f'{g}' for g in labels]
df_full[cols_edad] = df_full[cols_edad].fillna(0).astype(int)

df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,0,0,0,0
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,0,0,0,0
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,0,0,0,0
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,0,1,0,0
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,0,2,0,0


#### **1.7.2.4. Número de Casos por Regimen de Salud**

In [48]:
df.columns

Index(['ano', 'area', 'edad', 'sexo', 'semana', 'departamento_notificacion',
       'departamento_ocurrencia', 'municipio_ocurrencia', 'tip_cas',
       'confirmado', 'pac_hos', 'fec_not', 'ini_sin', 'fec_con', 'per_etn',
       'tip_ss', 'estado_final_de_caso', 'nombre_evento', 'uni_med',
       'cod_pais_o', 'consecutive', 'pais_ocurrencia', 'cod_dpto_o',
       'departamento', 'edad_anos', 'grupo_etario'],
      dtype='object')

In [49]:
df['tip_ss'].value_counts()

tip_ss
Subsidiado       370474
Contributivo     364747
Excepción         35076
No asegurado      20075
Especial          11574
Indeterminado      4344
Name: count, dtype: int64

In [50]:
df['tip_ss'] = df['tip_ss'].map({
    'Contributivo':'Contributivo',
    'Subsidiado':'Subsidiado',
    'Excepción': 'Subsidiado',
    'Especial': 'Subsidiado',
    'Indeterminado': 'No asegurado',
    'No asegurado':'No asegurado'
}).astype('category')

In [51]:
df['tip_ss'].value_counts()

tip_ss
Subsidiado      417124
Contributivo    364747
No asegurado     24419
Name: count, dtype: int64

In [52]:
tipo_ss_counts = (
    df.groupby(['ano', 'semana', 'departamento','tip_ss'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
tipo_ss_counts = tipo_ss_counts.rename(columns={'Subsidiado': 'casos_Subsidiado','Contributivo': 'casos_Contributivo','No asegurado':'casos_No_asegurado'})

In [53]:
df_full = df_full.merge(
    tipo_ss_counts,
    on=['ano', 'semana', 'departamento'],
    how='left'
)

In [54]:
cols_conteo = ['casos_Subsidiado','casos_Contributivo','casos_No_asegurado']
df_full[cols_conteo] = df_full[cols_conteo].fillna(0).astype(int)

In [55]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,0,0,0,0,0,0,0
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,0,0,0,0,0,0,0
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,0,0,0,0,0,0,0
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,0,1,0,0,0,0,1
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,0,2,0,0,1,1,1


#### **1.7.2.5. Población Anual**

In [56]:
df_pob = pd.read_excel("/home/guirlessa/Dl_Proyecto_Dengue/Covariables/poblacion_dpto_dengue_zona.xlsx")
df_pob.head()

,dpto_ccdgo,DPNOM,ANO,AREA_GEOGRAFICA,Total,Hombres,Mujeres,menor_13,14_26,27_59,mayor_59
0,5,Antioquia,2008,Cabecera Municipal,4243194,1995677,2247517,980534,1005028,1819186,472227
1,5,Antioquia,2008,Centros Poblados y Rural Disperso,1418905,734828,684077,442980,317746,530195,137605
2,5,Antioquia,2008,Total,5662099,2730505,2931594,1423514,1322774,2349381,609832
3,5,Antioquia,2009,Cabecera Municipal,4312348,2029876,2282472,971422,1022318,1860355,493569
4,5,Antioquia,2009,Centros Poblados y Rural Disperso,1418029,734359,683670,432820,318784,534546,141794


In [57]:
df_pob.rename(columns={
    'DPNOM': 'departamento',
    'ANO': 'ano',
    'Total': 'poblacion'}, inplace=True)
df_pob.head()

,dpto_ccdgo,departamento,ano,AREA_GEOGRAFICA,poblacion,Hombres,Mujeres,menor_13,14_26,27_59,mayor_59
0,5,Antioquia,2008,Cabecera Municipal,4243194,1995677,2247517,980534,1005028,1819186,472227
1,5,Antioquia,2008,Centros Poblados y Rural Disperso,1418905,734828,684077,442980,317746,530195,137605
2,5,Antioquia,2008,Total,5662099,2730505,2931594,1423514,1322774,2349381,609832
3,5,Antioquia,2009,Cabecera Municipal,4312348,2029876,2282472,971422,1022318,1860355,493569
4,5,Antioquia,2009,Centros Poblados y Rural Disperso,1418029,734359,683670,432820,318784,534546,141794


In [58]:
df_pob['departamento'].unique()

array(['Antioquia', 'Atlántico', 'Bogotá, D.C.', 'Bolívar', 'Boyacá',
       'Caldas', 'Caquetá', 'Cauca', 'Cesar', 'Córdoba', 'Cundinamarca',
       'Chocó', 'Huila', 'La Guajira', 'Magdalena', 'Meta', 'Nariño',
       'Norte de Santander', 'Quindio', 'Quindío', 'Risaralda',
       'Santander', 'Sucre', 'Tolima', 'Valle del Cauca', 'Arauca',
       'Casanare', 'Putumayo', 'Archipiélago de San Andrés',
       'Archipiélago de San Andrés, Providencia y Santa Catalina',
       'Amazonas', 'Guainía', 'Guaviare', 'Vaupés', 'Vichada'],
      dtype=object)

In [59]:
df_pob['departamento'] = df_pob['departamento'].apply(limpiar_nombre)

In [60]:
set(df_pob['departamento']) - set(df_full['departamento'])

{'Archipielago de san andres',
 'Archipielago de san andres, providencia y santa catalina',
 'Bogota, d.c.',
 'La guajira',
 'Norte de santander',
 'Valle del cauca'}

In [61]:
set(df_full['departamento']) - set(df_pob['departamento'])

{'Bogota d.c.',
 'Guajira',
 'Norte De Santander',
 'San Andres Y Providencia',
 'Valle Del Cauca'}

In [62]:
mapeo = {
    'La guajira': 'Guajira',
    'Archipielago de san andres': 'San Andres Y Providencia',
    'Archipielago de san andres, providencia y santa catalina': 'San Andres Y Providencia',
    'Bogota, d.c.': 'Bogota d.c.',
    'Norte de santander': 'Norte De Santander',
    'Valle del cauca': 'Valle Del Cauca'
}

In [63]:
df_pob['departamento'] = df_pob['departamento'].replace(mapeo)

In [64]:
set(df_full['departamento'].unique()) - set(df_pob['departamento'].unique())

set()

In [65]:
set(df_pob['departamento'].unique()) - set(df_full['departamento'].unique())

set()

In [66]:
poblacion = df_pob[df_pob['AREA_GEOGRAFICA'] == 'Total'].copy()

In [67]:
df_full = df_full.merge(
    poblacion[["departamento","ano", "poblacion"]],
    on=["departamento", "ano"],
    how="left"
)

In [68]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,0,0,0,0,0,0,0,65991.0
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,0,0,0,0,0,0,0,67361.0
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,0,0,0,0,0,0,0,67361.0
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,0,1,0,0,0,0,1,67361.0
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,0,2,0,0,1,1,1,67361.0


#### **1.7.2.6. Proporción Población Rural & Urbana**

In [69]:
df_pob['AREA_GEOGRAFICA'] = df_pob['AREA_GEOGRAFICA'].replace({
    'Cabecera Municipal': 'Urbano',
    'Centros Poblados y Rural Disperso': 'Rural'
})

df_pob_rural = df_pob[df_pob["AREA_GEOGRAFICA"] == "Rural"][["departamento","ano","poblacion"]].rename(columns={"poblacion":"poblacion_rural"})
df_pob_urbano = df_pob[df_pob["AREA_GEOGRAFICA"] == "Urbano"][["departamento","ano","poblacion"]].rename(columns={"poblacion":"poblacion_urbana"})

df_pob_area = pd.merge(
    df_pob_rural, 
    df_pob_urbano, 
    on=["departamento","ano"], 
    how="outer"
)

df_full = df_full.merge(
    df_pob_area,
    on=["departamento","ano"],
    how="left"
)

# Proporciones
df_full["prop_rural"]  = df_full["poblacion_rural"] / df_full["poblacion"]
df_full["prop_urbana"] = df_full["poblacion_urbana"] / df_full["poblacion"]

df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],...,(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion,poblacion_rural,poblacion_urbana,prop_rural,prop_urbana
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,...,0,0,0,0,0,65991.0,33964.0,32027.0,0.514676,0.485324
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,...,0,0,0,0,0,67361.0,34706.0,32655.0,0.515224,0.484776
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,...,0,0,0,0,0,67361.0,34706.0,32655.0,0.515224,0.484776
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,...,0,0,0,0,1,67361.0,34706.0,32655.0,0.515224,0.484776
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,...,0,0,1,1,1,67361.0,34706.0,32655.0,0.515224,0.484776


#### **1.7.2.7. Contrucción del Target - Incidenciax100.000 hab**

In [70]:
df_full["incidencia"] = (df_full["casos"] / df_full["poblacion"]) * 100000
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],...,60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion,poblacion_rural,poblacion_urbana,prop_rural,prop_urbana,incidencia
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,...,0,0,0,0,65991.0,33964.0,32027.0,0.514676,0.485324,0.000000
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,...,0,0,0,0,67361.0,34706.0,32655.0,0.515224,0.484776,0.000000
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,...,0,0,0,0,67361.0,34706.0,32655.0,0.515224,0.484776,0.000000
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,...,0,0,0,1,67361.0,34706.0,32655.0,0.515224,0.484776,1.484539
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,...,0,1,1,1,67361.0,34706.0,32655.0,0.515224,0.484776,4.453616


#### **1.7.2.8. Lags Incidencia**

**Lags:** Son variables que representan valores pasados de una serie temporal. su objetivo es incorporar memoria temporal al modelo.

In [71]:
# Lista de rezagos que quieres calcular
lags = [1, 2, 3, 4, 8, 12,20]  # puedes poner los que necesites

# Generar las columnas de lags
for lag in lags:
    df_full[f"inc_lag_{lag}"] = (
        df_full.groupby("departamento")["incidencia"].shift(lag)
    )

In [72]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],...,prop_rural,prop_urbana,incidencia,inc_lag_1,inc_lag_2,inc_lag_3,inc_lag_4,inc_lag_8,inc_lag_12,inc_lag_20
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,...,0.514676,0.485324,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,...,0.515224,0.484776,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,...,0.515224,0.484776,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,...,0.515224,0.484776,1.484539,0.000000,0.0,0.0,NaN,NaN,NaN,NaN
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,...,0.515224,0.484776,4.453616,1.484539,0.0,0.0,0.0,NaN,NaN,NaN


### **1.7.3. Feature Enginering Covariables**

Se añaden las covariables `temperatura`, `huemdad` y `precipitaciones`. La información de estas variables fueron estraidas por medio de la API de Google Earth Engine. Se hace uso de esta fuente de datos debido a su uso frecuente en la literatura, tanto para investigaciones nacionales como internacionales, lo que hace a esta una fuente de datos confiable. 

#### **1.7.3.1. Humedad**

La variable de humedad se encuentra expresada en porcentaje (%).

In [73]:
df_hum = pd.read_csv("/home/guirlessa/Dl_Proyecto_Dengue/Covariables/humedad_relativa_semanal_2011_2024.csv", index_col=0)
df_hum.head()

,dpto_ccdgo,dpto_cnmbr,hr_pct,unidad,week_end,week_num,week_start,year,.geo
system:index,,,,,,,,,
0_1_00000000000000000004,15,BOYACÁ,82.077662,porcentaje,2011-01-07,52,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_1_0000000000000000000c,41,HUILA,81.359597,porcentaje,2011-01-07,52,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_1_00000000000000000016,73,TOLIMA,82.836797,porcentaje,2011-01-07,52,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_1_00000000000000000013,66,RISARALDA,91.994325,porcentaje,2011-01-07,52,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_1_00000000000000000006,18,CAQUETÁ,77.070546,porcentaje,2011-01-07,52,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [74]:
df_hum["week_start"] = pd.to_datetime(df_hum["week_start"])
iso = df_hum["week_start"].dt.isocalendar()
df_hum["ano"] = iso.year.astype("int64")
df_hum["semana"] = iso.week.astype("int64")

df_hum = df_hum.rename(columns={
    "dpto_cnmbr": "departamento",
    "hr_pct": "humedad_relativa"
})

In [75]:
df_hum["departamento"] = df_hum["departamento"].apply(limpiar_nombre)
map_departamentos = {
    "La guajira": "Guajira",
    "Archipielago de san andres, providencia y santa catalina": "San Andres Y Providencia",
    "Bogota, d.c.": "Bogota d.c.",
    "Norte de santander": "Norte De Santander",
    "Valle del cauca": "Valle Del Cauca"
}
df_hum["departamento"] = df_hum["departamento"].replace(map_departamentos)

In [76]:
df_hum = df_hum[(df_hum["ano"] >= 2011) & (df_hum["ano"] <= 2024)].copy()

In [77]:
df_hum = df_hum.sort_values(["departamento","ano","semana"])
lags = [1,2,3,4,8,12,20]
for lag in lags:
    df_hum[f"humedad_lag_{lag}"] = (
        df_hum.groupby("departamento")["humedad_relativa"].shift(lag)
    )

In [78]:
df_full["ano"]    = df_full["ano"].astype("int64")
df_full["semana"] = df_full["semana"].astype("int64")

df_full = df_full.merge(
    df_hum[["departamento","ano","semana","humedad_relativa"] + [f"humedad_lag_{lag}" for lag in lags]],
    on=["departamento","ano","semana"],
    how="left"
)

In [79]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],...,inc_lag_12,inc_lag_20,humedad_relativa,humedad_lag_1,humedad_lag_2,humedad_lag_3,humedad_lag_4,humedad_lag_8,humedad_lag_12,humedad_lag_20
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,...,NaN,NaN,89.404425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,...,NaN,NaN,88.471605,89.404425,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,...,NaN,NaN,83.788504,88.471605,89.404425,NaN,NaN,NaN,NaN,NaN
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,...,NaN,NaN,90.717304,83.788504,88.471605,89.404425,NaN,NaN,NaN,NaN


#### **1.7.3.2. Precipitaciones**

In [80]:
df_precip = pd.read_csv("/home/guirlessa/Dl_Proyecto_Dengue/Covariables/precip_semanal_deptos_2011_2024.csv", index_col=0)
df_precip.head()

,dpto_ccdgo,dpto_cnmbr,precip_mm,unidad,week_end,week_num,week_start,year,.geo
system:index,,,,,,,,,
0_0_00000000000000000004,15,BOYACÁ,10.715501,mm,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_0000000000000000000c,41,HUILA,44.750945,mm,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_00000000000000000016,73,TOLIMA,42.865915,mm,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_00000000000000000013,66,RISARALDA,13.267050,mm,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_00000000000000000006,18,CAQUETÁ,8.582469,mm,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [81]:
df_precip = df_precip.rename(columns = {
     'dpto_cnmbr': 'departamento',
    'precip_mm': 'precipitacion',
    'year': 'ano',
    'week_num': 'semana'
})

In [82]:
df_precip['semana'] = df_precip['semana'].astype(int)

In [83]:
df_precip['departamento'] = df_precip['departamento'].apply(limpiar_nombre)
df_precip['departamento'].unique()

array(['Boyaca', 'Huila', 'Tolima', 'Risaralda', 'Caqueta', 'Casanare',
       'Putumayo', 'Amazonas', 'Guainia', 'Guaviare', 'Vaupes', 'Vichada',
       'Bogota, d.c.', 'Cauca', 'Antioquia', 'Bolivar', 'Cundinamarca',
       'Narino', 'Caldas', 'Atlantico', 'Norte de santander', 'Santander',
       'Valle del cauca', 'Choco', 'Cordoba', 'Meta', 'La guajira',
       'Magdalena', 'Quindio', 'Sucre', 'Cesar', 'Arauca',
       'Archipielago de san andres, providencia y santa catalina'],
      dtype=object)

In [84]:
df_precip['departamento'] = df_precip['departamento'].replace(map_departamentos)
df_precip['departamento'].unique()

array(['Boyaca', 'Huila', 'Tolima', 'Risaralda', 'Caqueta', 'Casanare',
       'Putumayo', 'Amazonas', 'Guainia', 'Guaviare', 'Vaupes', 'Vichada',
       'Bogota d.c.', 'Cauca', 'Antioquia', 'Bolivar', 'Cundinamarca',
       'Narino', 'Caldas', 'Atlantico', 'Norte De Santander', 'Santander',
       'Valle Del Cauca', 'Choco', 'Cordoba', 'Meta', 'Guajira',
       'Magdalena', 'Quindio', 'Sucre', 'Cesar', 'Arauca',
       'San Andres Y Providencia'], dtype=object)

In [85]:
set(df_precip['departamento']) - set(df_full['departamento'])

set()

In [86]:
set(df_full['departamento']) - set(df_precip['departamento'])

set()

In [87]:
df_precip = df_precip[
    (
        (df_precip["ano"] >= 2011) &
        (df_precip["ano"] <= 2024)
    )
    |
    (
        (df_precip["ano"] == 2011) &
        (df_precip["semana"] >= 1)
    )
].copy()

In [88]:
df_precip["ano"] = df_precip["ano"].astype(int)
df_precip["semana"] = df_precip["semana"].astype(int)

In [89]:
df_precip = df_precip.sort_values(
    ["departamento", "ano", "semana"]
)

In [90]:
for lag in [1, 2, 3, 4,8,12,20]:
    df_precip[f"precip_lag_{lag}"] = (
        df_precip
        .groupby("departamento")["precipitacion"]
        .shift(lag)
    )
    
df_precip.head()

,dpto_ccdgo,departamento,precipitacion,unidad,week_end,semana,week_start,ano,.geo,precip_lag_1,precip_lag_2,precip_lag_3,precip_lag_4,precip_lag_8,precip_lag_12,precip_lag_20
system:index,,,,,,,,,,,,,,,,
0_0_0000000000000000001c,91,Amazonas,18.358061,mm,2011-01-07,1,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",NaN,NaN,NaN,NaN,NaN,NaN,NaN
0_1_0000000000000000001c,91,Amazonas,73.413089,mm,2011-01-14,2,2011-01-08,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",18.358061,NaN,NaN,NaN,NaN,NaN,NaN
0_2_0000000000000000001c,91,Amazonas,57.676689,mm,2011-01-21,3,2011-01-15,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",73.413089,18.358061,NaN,NaN,NaN,NaN,NaN
0_3_0000000000000000001c,91,Amazonas,52.873876,mm,2011-01-28,4,2011-01-22,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",57.676689,73.413089,18.358061,NaN,NaN,NaN,NaN
0_4_0000000000000000001c,91,Amazonas,68.568418,mm,2011-02-04,5,2011-01-29,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",52.873876,57.676689,73.413089,18.358061,NaN,NaN,NaN


In [91]:
df_full = df_full.merge(
    df_precip[
        [
            "departamento",
            "ano",
            "semana",
            "precipitacion",
            "precip_lag_1",
            "precip_lag_2",
            "precip_lag_3",
            "precip_lag_4",
            "precip_lag_8",
            "precip_lag_12",
            "precip_lag_20"
        ]
    ],
    on=["departamento", "ano", "semana"],
    how="left"
)

df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],...,humedad_lag_12,humedad_lag_20,precipitacion,precip_lag_1,precip_lag_2,precip_lag_3,precip_lag_4,precip_lag_8,precip_lag_12,precip_lag_20
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,...,NaN,NaN,18.358061,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,...,NaN,NaN,73.413089,18.358061,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,...,NaN,NaN,57.676689,73.413089,18.358061,NaN,NaN,NaN,NaN,NaN
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,...,NaN,NaN,52.873876,57.676689,73.413089,18.358061,NaN,NaN,NaN,NaN


#### **1.7.3.3. Temperatura**

In [92]:
df_temp = pd.read_csv("/home/guirlessa/Dl_Proyecto_Dengue/Covariables/temp_semanal_deptos_2011_2024.csv", index_col=0)
df_temp.head()

,dpto_ccdgo,dpto_cnmbr,temp_c,unidad,week_end,week_num,week_start,year,.geo
system:index,,,,,,,,,
0_0_00000000000000000004,15,BOYACÁ,14.077837,celsius,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_0000000000000000000c,41,HUILA,17.908212,celsius,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_00000000000000000016,73,TOLIMA,17.068705,celsius,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_00000000000000000013,66,RISARALDA,16.630462,celsius,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"
0_0_00000000000000000006,18,CAQUETÁ,24.878685,celsius,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [93]:
df_temp = df_temp.rename(columns = {
     'dpto_cnmbr': 'departamento',
    'precip_mm': 'precipitacion',
    'year': 'ano',
    'week_num': 'semana'
})
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24486 entries, 0_0_00000000000000000004 to 13_52_0000000000000000001b
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   dpto_ccdgo    24486 non-null  int64  
 1   departamento  24486 non-null  object 
 2   temp_c        23744 non-null  float64
 3   unidad        24486 non-null  object 
 4   week_end      24486 non-null  object 
 5   semana        24486 non-null  float64
 6   week_start    24486 non-null  object 
 7   ano           24486 non-null  int64  
 8   .geo          24486 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 1.9+ MB


In [94]:
df_temp['departamento'] = df_temp['departamento'].apply(limpiar_nombre)
df_temp['departamento'] = df_temp['departamento'].replace(map_departamentos)

df_temp['departamento'].unique()

array(['Boyaca', 'Huila', 'Tolima', 'Risaralda', 'Caqueta', 'Casanare',
       'Putumayo', 'Amazonas', 'Guainia', 'Guaviare', 'Vaupes', 'Vichada',
       'Bogota d.c.', 'Cauca', 'Antioquia', 'Bolivar', 'Cundinamarca',
       'Narino', 'Caldas', 'Atlantico', 'Norte De Santander', 'Santander',
       'Valle Del Cauca', 'Choco', 'Cordoba', 'Meta', 'Guajira',
       'Magdalena', 'Quindio', 'Sucre', 'Cesar', 'Arauca',
       'San Andres Y Providencia'], dtype=object)

In [95]:
df_temp = df_temp[
    (
        (df_temp["ano"] >= 2011) &
        (df_temp["ano"] <= 2024)
    )
    |
    (
        (df_temp["ano"] == 2011) &
        (df_temp["semana"] >= 1)
    )
].copy()

In [96]:
df_temp = df_temp.sort_values(
    ["departamento", "ano", "semana"]
)

In [97]:
for lag in [1, 2, 3, 4,8,12,20]:
    df_temp[f"temp_lag_{lag}"] = (
        df_temp
        .groupby("departamento")["temp_c"]
        .shift(lag)
    )
    
df_temp.head()

,dpto_ccdgo,departamento,temp_c,unidad,week_end,semana,week_start,ano,.geo,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_8,temp_lag_12,temp_lag_20
system:index,,,,,,,,,,,,,,,,
0_0_0000000000000000001c,91,Amazonas,25.804489,celsius,2011-01-07,1.0,2011-01-01,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",NaN,NaN,NaN,NaN,NaN,NaN,NaN
0_1_0000000000000000001c,91,Amazonas,25.019756,celsius,2011-01-14,2.0,2011-01-08,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",25.804489,NaN,NaN,NaN,NaN,NaN,NaN
0_2_0000000000000000001c,91,Amazonas,25.395640,celsius,2011-01-21,3.0,2011-01-15,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",25.019756,25.804489,NaN,NaN,NaN,NaN,NaN
0_3_0000000000000000001c,91,Amazonas,25.464992,celsius,2011-01-28,4.0,2011-01-22,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",25.395640,25.019756,25.804489,NaN,NaN,NaN,NaN
0_4_0000000000000000001c,91,Amazonas,24.626118,celsius,2011-02-04,5.0,2011-01-29,2011,"{""type"":""MultiPoint"",""coordinates"":[]}",25.464992,25.395640,25.019756,25.804489,NaN,NaN,NaN


In [98]:
df_full = df_full.merge(
    df_temp[
        [
            "departamento",
            "ano",
            "semana",
            "temp_c",
            "temp_lag_1",
            "temp_lag_2",
            "temp_lag_3",
            "temp_lag_4",
            "temp_lag_8",
            "temp_lag_12",
            "temp_lag_20"
        ]
    ],
    on=["departamento", "ano", "semana"],
    how="left"
)

pd.set_option('display.max_columns', None)
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion,poblacion_rural,poblacion_urbana,prop_rural,prop_urbana,incidencia,inc_lag_1,inc_lag_2,inc_lag_3,inc_lag_4,inc_lag_8,inc_lag_12,inc_lag_20,humedad_relativa,humedad_lag_1,humedad_lag_2,humedad_lag_3,humedad_lag_4,humedad_lag_8,humedad_lag_12,humedad_lag_20,precipitacion,precip_lag_1,precip_lag_2,precip_lag_3,precip_lag_4,precip_lag_8,precip_lag_12,precip_lag_20,temp_c,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_8,temp_lag_12,temp_lag_20
0,Amazonas,2011-01-01,2010,52,0.0,0,0,0,0,0,0,0,0,0,0,0,0,65991.0,33964.0,32027.0,0.514676,0.485324,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazonas,2011-01-03,2011,1,0.0,0,0,0,0,0,0,0,0,0,0,0,0,67361.0,34706.0,32655.0,0.515224,0.484776,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,89.404425,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.358061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.804489,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Amazonas,2011-01-10,2011,2,0.0,0,0,0,0,0,0,0,0,0,0,0,0,67361.0,34706.0,32655.0,0.515224,0.484776,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,88.471605,89.404425,NaN,NaN,NaN,NaN,NaN,NaN,73.413089,18.358061,NaN,NaN,NaN,NaN,NaN,NaN,25.019756,25.804489,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazonas,2011-01-17,2011,3,1.0,0,1,0,1,0,0,1,0,0,0,0,1,67361.0,34706.0,32655.0,0.515224,0.484776,1.484539,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,83.788504,88.471605,89.404425,NaN,NaN,NaN,NaN,NaN,57.676689,73.413089,18.358061,NaN,NaN,NaN,NaN,NaN,25.395640,25.019756,25.804489,NaN,NaN,NaN,NaN,NaN
4,Amazonas,2011-01-24,2011,4,3.0,1,2,0,3,1,0,2,0,0,1,1,1,67361.0,34706.0,32655.0,0.515224,0.484776,4.453616,1.484539,0.0,0.0,0.0,NaN,NaN,NaN,90.717304,83.788504,88.471605,89.404425,NaN,NaN,NaN,NaN,52.873876,57.676689,73.413089,18.358061,NaN,NaN,NaN,NaN,25.464992,25.395640,25.019756,25.804489,NaN,NaN,NaN,NaN


### **1.7.4. Verificación Valores Faltantes**

In [99]:
Resumen_Faltantes(df_full)

,faltantes,porcentaje
humedad_lag_20,1468,6.077
precip_lag_20,1436,5.945
temp_lag_20,1436,5.945
humedad_lag_12,1212,5.017
precip_lag_12,1180,4.885
temp_lag_12,1180,4.885
humedad_lag_8,1084,4.487
temp_lag_8,1052,4.355
precip_lag_8,1052,4.355
humedad_lag_4,956,3.958


In [100]:
df_full = df_full[
    (df_full['ano'] >= 2012) &
    (df_full['ano'] <= 2024)
]

In [101]:
# porcentaje de faltantes por departamento
faltantes_depto = (
    df_full
    .isna()
    .groupby(df_full['departamento'])
    .sum()
)

faltantes_depto.round(2)

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion,poblacion_rural,poblacion_urbana,prop_rural,prop_urbana,incidencia,inc_lag_1,inc_lag_2,inc_lag_3,inc_lag_4,inc_lag_8,inc_lag_12,inc_lag_20,humedad_relativa,humedad_lag_1,humedad_lag_2,humedad_lag_3,humedad_lag_4,humedad_lag_8,humedad_lag_12,humedad_lag_20,precipitacion,precip_lag_1,precip_lag_2,precip_lag_3,precip_lag_4,precip_lag_8,precip_lag_12,precip_lag_20,temp_c,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_8,temp_lag_12,temp_lag_20
departamento,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Amazonas,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Antioquia,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Arauca,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Atlantico,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Bogota d.c.,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Bolivar,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Boyaca,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Caldas,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Caqueta,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Se observa que el departamento de San Andrés y Providencia presenta un 100 % de valores faltantes en las covariables consideradas en el estudio. Debido a la ausencia total de información complementaria, se decidió excluir este departamento del análisis con el fin de garantizar la consistencia y viabilidad metodológica del estudio.

In [108]:
df_full = df_full[df_full["departamento"] != "San Andres Y Providencia"].copy()

In [109]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion,poblacion_rural,poblacion_urbana,prop_rural,prop_urbana,incidencia,inc_lag_1,inc_lag_2,inc_lag_3,inc_lag_4,inc_lag_8,inc_lag_12,inc_lag_20,humedad_relativa,humedad_lag_1,humedad_lag_2,humedad_lag_3,humedad_lag_4,humedad_lag_8,humedad_lag_12,humedad_lag_20,precipitacion,precip_lag_1,precip_lag_2,precip_lag_3,precip_lag_4,precip_lag_8,precip_lag_12,precip_lag_20,temp_c,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_8,temp_lag_12,temp_lag_20
53,Amazonas,2012-01-02,2012,1,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.484539,90.290603,91.799789,93.952762,93.012129,90.353373,88.268190,86.796777,85.665839,77.331730,15.992617,43.339945,99.604370,82.702236,56.271440,50.983546,39.108242,24.848668,24.890654,24.334752,24.545618,25.345678,25.278160,25.874930,24.364443
54,Amazonas,2012-01-09,2012,2,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,91.483124,90.290603,91.799789,93.952762,93.012129,90.031361,88.516930,89.352117,111.668209,77.331730,15.992617,43.339945,99.604370,61.414601,61.818291,62.830250,24.729721,24.848668,24.890654,24.334752,24.545618,25.084141,25.495665,24.971798
55,Amazonas,2012-01-16,2012,3,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.484539,88.076963,91.483124,90.290603,91.799789,93.952762,90.432841,87.642961,88.603154,46.687589,111.668209,77.331730,15.992617,43.339945,93.713083,24.950393,42.988018,24.638674,24.729721,24.848668,24.890654,24.334752,25.751070,25.760539,25.304947
56,Amazonas,2012-01-23,2012,4,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.484539,91.751947,88.076963,91.483124,90.290603,91.799789,90.054035,89.807275,89.829683,49.829880,46.687589,111.668209,77.331730,15.992617,40.426032,88.169343,56.572467,25.164008,24.638674,24.729721,24.848668,24.890654,25.284157,25.842089,25.166142
57,Amazonas,2012-01-30,2012,5,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,94.732196,91.751947,88.076963,91.483124,90.290603,90.353373,88.268190,89.164113,84.010583,49.829880,46.687589,111.668209,77.331730,82.702236,56.271440,74.893047,24.604644,25.164008,24.638674,24.729721,24.848668,25.345678,25.278160,25.312395


In [110]:
Resumen_Faltantes(df_full)

,faltantes,porcentaje
humedad_relativa,32,0.147
humedad_lag_1,32,0.147
humedad_lag_2,32,0.147
humedad_lag_3,32,0.147
humedad_lag_4,32,0.147
humedad_lag_8,32,0.147
humedad_lag_12,32,0.147
humedad_lag_20,32,0.147


In [111]:
faltantes_df = df_full[df_full.isna().any(axis=1)]

faltantes_df.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion,poblacion_rural,poblacion_urbana,prop_rural,prop_urbana,incidencia,inc_lag_1,inc_lag_2,inc_lag_3,inc_lag_4,inc_lag_8,inc_lag_12,inc_lag_20,humedad_relativa,humedad_lag_1,humedad_lag_2,humedad_lag_3,humedad_lag_4,humedad_lag_8,humedad_lag_12,humedad_lag_20,precipitacion,precip_lag_1,precip_lag_2,precip_lag_3,precip_lag_4,precip_lag_8,precip_lag_12,precip_lag_20,temp_c,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_8,temp_lag_12,temp_lag_20
730,Amazonas,2024-12-23,2024,52,0.0,0,0,0,0,0,0,0,0,0,0,0,0,84109.0,42017.0,42092.0,0.499554,0.500446,0.000000,2.377867,1.188933,3.566800,5.944667,2.377867,1.188933,2.377867,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.894767,91.363304,44.367803,62.185759,53.916425,69.257458,22.791240,110.581182,25.356219,25.436733,25.740608,26.217013,26.191706,26.768288,27.237145,25.930929
1462,Antioquia,2024-12-23,2024,52,146.0,71,75,38,108,10,28,38,51,19,72,3,71,6880799.0,1477084.0,5403715.0,0.214668,0.785332,2.121847,2.470643,3.589699,3.560633,3.400768,4.461691,4.476224,5.958610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.800715,44.854276,15.221467,23.887855,50.458445,81.880910,55.542877,59.588436,20.901588,21.741482,21.893370,21.660297,21.726839,21.550773,21.626891,21.807521
2194,Arauca,2024-12-23,2024,52,11.0,7,4,5,6,0,5,2,3,1,2,1,8,277883.0,97697.0,180186.0,0.351576,0.648424,3.958501,7.557137,7.557137,6.477546,4.318364,3.598637,6.837410,7.917001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.785270,11.149117,0.216387,0.213313,5.997191,60.860407,62.749647,69.744724,25.951580,26.047516,25.426645,25.486402,25.750842,25.229128,25.586758,24.840169
2926,Atlantico,2024-12-23,2024,52,433.0,210,223,4,429,23,145,183,79,3,183,8,242,2836795.0,143928.0,2692867.0,0.050736,0.949264,15.263704,14.699687,13.818411,12.126361,8.707009,6.944457,5.604917,6.662448,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.465338,13.255229,0.000000,11.817061,30.061106,101.740268,48.229929,29.452350,26.699985,27.640162,27.285996,27.626163,26.961338,26.416725,27.138740,27.205589
3658,Bogota d.c.,2024-12-23,2024,52,51.0,31,20,13,38,2,6,6,24,13,39,2,10,7918660.0,29847.0,7888813.0,0.003769,0.996231,0.644048,0.416737,0.644048,0.454622,0.303082,0.492508,0.580906,0.820846,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.102271,5.899801,1.392379,0.567428,3.023635,33.130952,33.654872,40.544550,11.593668,11.632818,11.439406,11.998250,11.866570,11.879165,11.655969,11.973988


In [112]:
faltantes_df["departamento"].value_counts()

departamento
Amazonas              1
Antioquia             1
Arauca                1
Atlantico             1
Bogota d.c.           1
Bolivar               1
Boyaca                1
Caldas                1
Caqueta               1
Casanare              1
Cauca                 1
Cesar                 1
Choco                 1
Cordoba               1
Cundinamarca          1
Guainia               1
Guajira               1
Guaviare              1
Huila                 1
Magdalena             1
Meta                  1
Narino                1
Norte De Santander    1
Putumayo              1
Quindio               1
Risaralda             1
Santander             1
Sucre                 1
Tolima                1
Valle Del Cauca       1
Vaupes                1
Vichada               1
Name: count, dtype: int64

In [114]:
faltantes_df.groupby(["ano", "semana"]).size()

ano   semana
2024  52        32
dtype: int64

Se identificó la presencia de valores faltantes en las variables de humedad relativa y sus respectivos rezagos. El análisis evidenció que dichos faltantes corresponden exclusivamente a la semana epidemiológica 52 del año 2024, afectando únicamente un registro por departamento. Dado que representan un porcentaje mínimo del conjunto de datos y se concentran en el último periodo temporal del estudio, se decidió excluir estos registros para preservar la consistencia del análisis y evitar procesos de imputación que puedan introducir sesgos al análisis.


In [115]:
df_full = df_full[~(
    (df_full["ano"] == 2024) &
    (df_full["semana"] == 52)
)].copy()

In [116]:
df_full.head()

,departamento,fecha_inicio,ano,semana,casos,casos_mujeres,casos_hombres,casos_rural,casos_urbano,(0_5],(5_13],(13_26],(26_60],60_mas,casos_Contributivo,casos_No_asegurado,casos_Subsidiado,poblacion,poblacion_rural,poblacion_urbana,prop_rural,prop_urbana,incidencia,inc_lag_1,inc_lag_2,inc_lag_3,inc_lag_4,inc_lag_8,inc_lag_12,inc_lag_20,humedad_relativa,humedad_lag_1,humedad_lag_2,humedad_lag_3,humedad_lag_4,humedad_lag_8,humedad_lag_12,humedad_lag_20,precipitacion,precip_lag_1,precip_lag_2,precip_lag_3,precip_lag_4,precip_lag_8,precip_lag_12,precip_lag_20,temp_c,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_8,temp_lag_12,temp_lag_20
53,Amazonas,2012-01-02,2012,1,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.484539,90.290603,91.799789,93.952762,93.012129,90.353373,88.268190,86.796777,85.665839,77.331730,15.992617,43.339945,99.604370,82.702236,56.271440,50.983546,39.108242,24.848668,24.890654,24.334752,24.545618,25.345678,25.278160,25.874930,24.364443
54,Amazonas,2012-01-09,2012,2,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,91.483124,90.290603,91.799789,93.952762,93.012129,90.031361,88.516930,89.352117,111.668209,77.331730,15.992617,43.339945,99.604370,61.414601,61.818291,62.830250,24.729721,24.848668,24.890654,24.334752,24.545618,25.084141,25.495665,24.971798
55,Amazonas,2012-01-16,2012,3,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.484539,88.076963,91.483124,90.290603,91.799789,93.952762,90.432841,87.642961,88.603154,46.687589,111.668209,77.331730,15.992617,43.339945,93.713083,24.950393,42.988018,24.638674,24.729721,24.848668,24.890654,24.334752,25.751070,25.760539,25.304947
56,Amazonas,2012-01-23,2012,4,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.484539,91.751947,88.076963,91.483124,90.290603,91.799789,90.054035,89.807275,89.829683,49.829880,46.687589,111.668209,77.331730,15.992617,40.426032,88.169343,56.572467,25.164008,24.638674,24.729721,24.848668,24.890654,25.284157,25.842089,25.166142
57,Amazonas,2012-01-30,2012,5,0.0,0,0,0,0,0,0,0,0,0,0,0,0,68681.0,35413.0,33268.0,0.515616,0.484384,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,94.732196,91.751947,88.076963,91.483124,90.290603,90.353373,88.268190,89.164113,84.010583,49.829880,46.687589,111.668209,77.331730,82.702236,56.271440,74.893047,24.604644,25.164008,24.638674,24.729721,24.848668,25.345678,25.278160,25.312395


In [117]:
Resumen_Faltantes(df_full)

,faltantes,porcentaje


Finalmente, se genera la base de datos final consolidada y depurada, la cual queda preparada para las etapas posteriores de análisis exploratorio y modelado.


In [118]:
df_full.to_csv("/home/guirlessa/Dl_Proyecto_Dengue/Bases_Datos/df_full.csv", index=False)

## **1.8. Adicional: Limpiando Shapefile**

In [2]:
gdf = gpd.read_file("/home/guirlessa/Dl_Proyecto_Dengue/Covariables/MGN_ADM_DPTO_POLITICO/MGN_ADM_DPTO_POLITICO.shp")
gdf.head()

,dpto_ccdgo,dpto_cnmbr,dpto_ano_c,dpto_act_a,dpto_narea,dpto_nano,shape_Leng,shape_Area,geometry
0,05,ANTIOQUIA,1886,Constitucion Politica de 1886,62788.733068,2025,21.501928,5.135050,"POLYGON ((-76.41355 8.87383, -76.40465 8.85195..."
1,08,ATLÃNTICO,1910,Ley 21 de 1910,3324.295762,2025,2.711196,0.274826,"POLYGON ((-74.84946 11.09778, -74.84938 11.097..."
2,11,"BOGOTÃ, D.C.",1538,Constitucion Politica de 1886,1620.540834,2025,3.764120,0.132175,"POLYGON ((-74.05361 4.83292, -74.0536 4.83295,..."
3,13,BOLÃVAR,1886,Constitucion Politica de 1886,26680.806449,2025,16.389479,2.194558,"MULTIPOLYGON (((-76.1747 9.39857, -76.17467 9...."
4,15,BOYACÃ,1886,Constitucion Politica de 1886,23073.396960,2024,15.991512,1.886665,"POLYGON ((-72.18436 7.02894, -72.18409 7.02893..."


In [3]:
gdf.columns

Index(['dpto_ccdgo', 'dpto_cnmbr', 'dpto_ano_c', 'dpto_act_a', 'dpto_narea',
       'dpto_nano', 'shape_Leng', 'shape_Area', 'geometry'],
      dtype='object')

In [122]:
gdf['dpto_cnmbr'] = gdf['dpto_cnmbr'].apply(limpiar_nombre)
gdf['dpto_cnmbr'].unique()

array(['Antioquia', 'Atlantico', 'Bogota, d.c.', 'Bolavar', 'Boyaca',
       'Caldas', 'Caqueta', 'Cauca', 'Cesar', 'Cardoba', 'Cundinamarca',
       'Choca', 'Huila', 'La guajira', 'Magdalena', 'Meta', 'Nariao',
       'Norte de santander', 'Quindao', 'Risaralda', 'Santander', 'Sucre',
       'Tolima', 'Valle del cauca', 'Arauca', 'Casanare', 'Putumayo',
       'Archipialago de san andras, providencia y santa catalina',
       'Amazonas', 'Guainaa', 'Guaviare', 'Vaupas', 'Vichada'],
      dtype=object)

In [123]:
mapeo_gdf = {
    "La guajira": "Guajira",
    "Archipialago de san andras, providencia y santa catalina": "San Andres Y Providencia",
    "Bogota, d.c.": "Bogota d.c.",
    "Norte de santander": "Norte De Santander",
    "Valle del cauca": "Valle Del Cauca",
    "Cardoba":"Cordoba",
    "Choca":"Choco",
    "Bolavar":"Bolivar",
    "Nariao":"Narino",
    "Guainaa":"Guainia",
    "Quindao":"Quindio",
    "Vaupas":"Vaupes"}

gdf['dpto_cnmbr'] = gdf['dpto_cnmbr'].replace(mapeo_gdf)
gdf['dpto_cnmbr'].unique()

array(['Antioquia', 'Atlantico', 'Bogota d.c.', 'Bolivar', 'Boyaca',
       'Caldas', 'Caqueta', 'Cauca', 'Cesar', 'Cordoba', 'Cundinamarca',
       'Choco', 'Huila', 'Guajira', 'Magdalena', 'Meta', 'Narino',
       'Norte De Santander', 'Quindio', 'Risaralda', 'Santander', 'Sucre',
       'Tolima', 'Valle Del Cauca', 'Arauca', 'Casanare', 'Putumayo',
       'San Andres Y Providencia', 'Amazonas', 'Guainia', 'Guaviare',
       'Vaupes', 'Vichada'], dtype=object)

In [124]:
set(df_full['departamento'].unique()) - set(gdf['dpto_cnmbr'].unique())

set()

In [125]:
set(gdf['dpto_cnmbr'].unique()) - set(df_full['departamento'].unique())

{'San Andres Y Providencia'}

Se procede a eliminar el departamento de *San Andrés y Providencia* del conjunto de datos, en concordancia con su exclusión previa de la base de datos final. Esta decisión garantiza la consistencia entre las distintas etapas del procesamiento y mantiene la homogeneidad del análisis a nivel departamental.

In [126]:
gdf = gdf[gdf['dpto_cnmbr'] != 'San Andres Y Providencia']

In [128]:
# Exportar a shapefile
gdf.to_file(
    "/home/guirlessa/Dl_Proyecto_Dengue/Covariables/MGN_ADM_DPTO_POLITICO/MGN_ADM_DPTO_POLITICO_limpio.shp",
    driver="ESRI Shapefile"
)